## **LSOUL — Weekly Report**



Re-runnable: every date derives from the parameters block. No hardcoded values.

**Ngôn ngữ:** toàn bộ phần nhận định (insights) luôn hiển thị bằng **tiếng Việt** — kể cả
phần dự phòng khi AI bị chặn. Nhãn cột và tiêu đề kỹ thuật giữ nguyên tiếng Anh theo quy ước
của dashboard.

---
## AI source: the LSOUL dashboard layer only

All narrative text comes from the dashboard's own AI layer and nothing else:

| Object | Supplies |
|---|---|
| `dash_ai_narrative` | `exec_summary_vi`, `narrative_vi` (highlights, lowlights, actions) |
| `dash_action_items` | specific owner-tagged actions, PERFORMANCE and STOCK |
| `dash_ai_review` | the quality gate — a failed narrative is not published |
| `dash_material_points` | the point set every field is anchored to — trade `P01…` **and marketing `M01…`** |
| `dash_trend_marketing_weekly` | paid media and website traffic, section 4 |

**Deliberately not used:** `report_performance_snapshot_ai_generated_v2`,
`ai_summary_gen_report_performance_snapshot`, `ai_product_stock_exec_summary`,
`ai_product_stock_insights`, `fn_stats_product_comments_ai_generated`. Those belong to
other reporting streams, are generated on their own schedules against their own
definitions, and mixing them in would put two different NMV bases and two different
materiality rules in one report.

### One consequence you should know about

**The dashboard AI layer produces no stock commentary prose.** It generates an executive
summary, highlights, lowlights and the Vietnamese action text — that is all. Section 5
therefore leads with the dashboard's STOCK action items, which are *deterministic SQL*,
not model output (the model only translates them into Vietnamese). They are specific and
SKU-anchored, so they read as commentary, but they are not AI prose and are labelled as
such.

---
## Definitions — confirmed against the data

| Term | As built | Note |
|---|---|---|
| **GMV** | Gross merchandise value, before cancellations/refunds | |
| **NMV** | Net. VN/TH natively net; China observed-net past 30 days, modelled `GMV x factor` inside 30 days | |
| **Bills** | Offline transactions, `level_1 = 'Store'` | |
| **Orders** | Online transactions, Ecommerce + Sale Online | |
| **AOV** | `NMV / (bills + orders)` | |
| **Hero** | Top 10 SKUs by units, year-to-date | code filters on the report year |
| **Weekly best seller** | Top 10 by units in report_week, heroes excluded | |
| **Slow mover** | Bottom 10 by sell-through, stock > 0 and bands below threshold | |
| **roas_channel** | `NMV / spend` for the channel | efficiency ratio, **not** incrementality |
| **delivery** | `COALESCE(impressions, reach)` | Google reports impressions, Meta/TikTok reach |

### What changed since the last version

**1. Targets now reach level_3.** `dash_trend_weekly` carries `week_target`, the level_2 plan
allocated by trailing-8-week NMV share (shares sum to 1, so it reconciles exactly). The old
note here said a level_3 target table was impossible — that is no longer true, and section 6's
level_3 breakdown now carries target columns.

**2. Marketing is in the report.** Paid media (Meta / TikTok / Google Ads) and GA4 website
traffic arrive via `dash_trend_marketing_weekly`. Material marketing movements appear as
`M01, M02…` alongside the trade `P##` series.

**3. The KPI target was wrong and is fixed.** `month_target_revenue` is a constant repeated on
every date row, so the old `SUM()` over a date range multiplied the monthly target by the number
of days — measured at **705.0bn instead of 23.5bn for August, a 30x overstatement**, which
pushed MTD attainment to about 3% when it was near 104%. Now read with `MAX()` per level_2, the
same way section 6's run-rate table already did.

**4. Section 5 was showing the wrong actions.** It queried `section='PERFORMANCE'`, so the stock
section displayed performance actions and the five STOCK items were never rendered anywhere.
STOCK items also carry `page='BOTH'`, which the page filter would not have matched.

### Still true, and still worth knowing
- **Thailand local-currency columns are inconsistent by 100x** — local is derived from VND at one
  documented rate per country; the `_thai` / `_cn` columns are never summed.
- **FX is hardcoded** (800 VND/THB, 3800 VND/CNY) inside the source functions.
- **Wholesales** is neither clearly offline nor online. Defaulted to *orders*; flip `WHOLESALE_CLASS`.
- **No year-on-year is possible** — `dash_base_order_day` starts 2026-01-01.

### Conventions applied
- Money: `<metric>_local | <metric>_vnd | currency_code`. Vietnam local == VND, not duplicated.
- Cross-country totals in **VND only**; local renders `—` on any total or multi-country row.
- Percentages to 1 decimal. Ranked tables sort descending by the ranking metric, ties broken by NMV.
- Number separators stay English throughout, matching the AI layer's HARD RULE 2.

---
## Parameters

In [ ]:
from google.cloud import bigquery
import pandas as pd, re, warnings; warnings.filterwarnings('ignore')
from IPython.display import HTML, display

PROJECT, DATASET = 'marine-cycle-484213-b4', 'reportdb'
FQ = f'{PROJECT}.{DATASET}'
client = bigquery.Client(project=PROJECT)

# ---- parameters -------------------------------------------------------------
REPORT_WEEK_END = None      # None = last completed week (Sunday), or 'YYYY-MM-DD'
TZ              = 'Asia/Ho_Chi_Minh'
RISK_THRESHOLD  = 1.30      # required run-rate above this x trailing-4wk avg = at risk
WHOLESALE_CLASS = 'order'   # 'order' or 'bill'
SHOW_QUARANTINED = True     # True = show failed AI text, clearly marked unverified
MONEY_DP, PCT_DP = 0, 1

# Insights are ALWAYS Vietnamese - including the deterministic fallback used when the AI
# layer is quarantined. There is no language toggle: an English fallback silently appearing
# inside a Vietnamese report was the failure mode this removes.
LANG = 'vi'

# Hardcoded inside the source functions; stated so the assumption is visible.
FX = {'VIETNAM': ('VND', 1.0), 'THAILAND': ('THB', 800.0), 'CHINA': ('CNY', 3800.0)}

BLUE, ORANGE, GREY, INK = '#1F4E9C', '#F5821F', '#8894A6', '#1B2430'
RED, GREEN = '#D9534F', '#28A745'

def q(sql, **kw):
    sql = sql.replace('{FQ}', FQ)
    for k, v in kw.items(): sql = sql.replace('{'+k+'}', str(v))
    return client.query(sql).to_dataframe()

if REPORT_WEEK_END:
    _wk = q("SELECT DATE('{d}') AS wk_end", d=REPORT_WEEK_END)
else:
    _wk = q("SELECT DATE_SUB(DATE_TRUNC(CURRENT_DATE('{tz}'), WEEK(MONDAY)), INTERVAL 1 DAY) AS wk_end", tz=TZ)

WK_END   = _wk.wk_end.iloc[0]
WK_START = WK_END - pd.Timedelta(days=6)
PW_END   = WK_START - pd.Timedelta(days=1)
PW_START = PW_END - pd.Timedelta(days=6)

# Grouped by (country, level_2), not level_2 alone. Channel names repeat across markets -
# 'VN Online' exists under both VIETNAM and THAILAND - so a level_2-only MIN() lets one
# market's late feed truncate another's.
cov = q("""
SELECT country, level_2,
       MIN(data_complete_through) AS complete_through,
       ANY_VALUE(load_note)       AS note
FROM `{FQ}.dash_dq_platform_day` GROUP BY country, level_2
""")
cov['complete_through'] = pd.to_datetime(cov.complete_through).dt.date
LAGGING = cov[cov.complete_through < WK_END]
WEEK_COMPLETE = LAGGING.empty

# How much loaded revenue the completeness cutoff is withholding this month. The old report
# said channels were "understated" without ever saying by how much.
WITHHELD = q("""
SELECT country, level_2, data_complete_through, days_covered, days_elapsed,
       nmv_excluded_by_cutoff
FROM `{FQ}.dash_trend_monthly`
WHERE is_current_month AND NOT is_month_data_complete
ORDER BY nmv_excluded_by_cutoff DESC
""")

# ---- period-comparison windows: DAYS-ELAPSED CAPPING -----------------------
# The report is "as of WK_END", so N is the day-of-month of WK_END, not of today.
#
# MONTH OVER MONTH. Month-to-date is compared against the SAME first N days of the
# prior month, never the prior month in full. Uncapped, at day 3 of 30, every
# channel printed -85% to -100% - that is 3 days measured against 30, not trade.
# Two explicit ranges, so neither side can silently drift:
#     current : MTD_START -> MTD_END   (day 1 .. N of this month)
#     prior   : PM_START  -> PM_END    (day 1 .. N of last month, clamped)
import datetime as _dt
_we       = pd.Timestamp(WK_END).date()
MTD_START = _we.replace(day=1)
MTD_END   = _we
N_DAYS    = (MTD_END - MTD_START).days + 1
_pm_last  = MTD_START - _dt.timedelta(days=1)          # last day of the prior month
PM_START  = _pm_last.replace(day=1)
# clamped: at N=31 against a 30-day prior month the window would otherwise spill
PM_END    = min(PM_START + _dt.timedelta(days=N_DAYS - 1), _pm_last)

# WEEK OVER WEEK. Already like-for-like - WK_START..WK_END and PW_START..PW_END are
# both full windows on the SAME week-start convention (Monday), which is unchanged.
# M is stated and asserted rather than assumed, so an edit to REPORT_WEEK_END that
# broke the symmetry would fail loudly instead of quietly skewing every comparison.
M_DAYS = (pd.Timestamp(WK_END).date() - pd.Timestamp(WK_START).date()).days + 1
assert (pd.Timestamp(PW_END).date() - pd.Timestamp(PW_START).date()).days + 1 == M_DAYS, \
    'WoW windows must cover the same number of days'

print(f'Tuần báo cáo : {WK_START} tới {WK_END}  ({M_DAYS} ngày)')
print(f'So sánh MoM (cùng kỳ): {MTD_START} tới {MTD_END}  vs  {PM_START} tới {PM_END}  (N={N_DAYS})')
print(f'Tuần trước   : {PW_START} tới {PW_END}  ({M_DAYS} ngày)')
#for _, r in LAGGING.iterrows():
#   print(f'   {r.country} / {r.level_2}: chỉ đủ dữ liệu tới {r.complete_through} — {r.note}')
#for _, r in WITHHELD.iterrows():
#    print(f'   [MTD] {r.country} / {r.level_2}: {r.nmv_excluded_by_cutoff/1e6:,.0f} triệu VND '
#          f'đã tải nhưng bị giữ lại ({r.days_covered}/{r.days_elapsed} ngày được đọc)')

In [ ]:
# ---- shared helpers ---------------------------------------------------------
TH = [{'selector':'th','props':[('background',BLUE),('color','#fff'),
       ('font','600 12px system-ui'),('padding','8px 10px'),('text-align','left')]},
      {'selector':'td','props':[('font','400 13px system-ui'),('padding','7px 10px'),
       ('border-bottom','1px solid #EEF1F5')]}]

def table(df, **kw):
    display(df.style.hide(axis='index').set_table_styles(TH, **kw))

def head(title, note=''):
    display(HTML(f'<h4 style="color:{BLUE};margin:18px 0 2px">{title}</h4>'
                 f'<div style="color:{GREY};font:400 12px system-ui;margin-bottom:8px">{note}</div>'))

def prose(txt, label, accent=BLUE, italic=False):
    if not txt or str(txt).strip() in ('','None'):
        display(HTML(f'<div style="color:{GREY};font:italic 13px system-ui">{label}: không có dữ liệu.</div>'))
        return
    style = 'font-style:italic;' if italic else ''
    display(HTML(f'''<div style="background:#fff;border:1px solid #E3E8EF;border-left:4px solid {accent};
      border-radius:6px;padding:14px 18px;margin:8px 0;font:400 14px/1.75 system-ui;color:{INK};{style}">
      <div style="font:600 11px system-ui;color:{GREY};text-transform:uppercase;letter-spacing:.06em;
      margin-bottom:8px;font-style:normal">{label}</div>{str(txt).replace(chr(10),'<br>')}</div>'''))

def bullets(items, accent, label):
    if not items:
        display(HTML(f'<div style="color:{GREY};font:italic 13px system-ui">Không có {label} trong kỳ này.</div>'))
        return
    html = ''.join(f'''<div style="background:#fff;border:1px solid #E3E8EF;border-left:4px solid {accent};
      border-radius:6px;padding:11px 16px;margin:6px 0;font:400 13.5px/1.65 system-ui;color:{INK}">{t}</div>'''
      for t in items)
    display(HTML(html))

def banner(text, accent=ORANGE):
    display(HTML(f'''<div style="background:#FFF6EC;border-left:4px solid {accent};padding:11px 15px;
      margin:10px 0;font:400 13px/1.6 system-ui;color:{INK}">{text}</div>'''))

def scalar(v):
    """ai_field can hand back a Series when dash_ai_narrative has duplicate pages."""
    if isinstance(v, (pd.Series, pd.DataFrame)):
        return v.iloc[0] if len(v) else ''
    return v

# The narrative is one run-on string with inline **Highlight -** / **Lowlight -**
# markers and an ACTIONS: tail, so split on the markers, not on newlines.
def split_narrative(text):
    if not text or not str(text).strip(): return [], [], ''
    parts   = re.split(r'\bACTIONS:\s*', str(text), maxsplit=1)
    body    = parts[0]
    actions = parts[1] if len(parts) > 1 else ''
    chunks  = re.split(r'\*\*(Highlight|Lowlight)\s*-\*\*', body)
    hi, lo = [], []
    for i in range(1, len(chunks)-1, 2):
        kind, txt = chunks[i], chunks[i+1].strip()
        if txt: (hi if kind == 'Highlight' else lo).append(txt)
    return hi, lo, actions.strip()

def _clean_action_item(item):
    """Drop blank list markers and stray ACTIONS: labels, keep owner text."""
    text = re.sub(r'^\s*[-*]\s*', '', str(item or '').strip())
    text = re.sub(r'^ACTIONS:\s*', '', text)
    if not text or re.fullmatch(r'\[.*?\]\s*-?\s*', text):
        return ''
    return text.strip()

def split_actions(txt):
    """Split ACTIONS on newlines or owner prefixes, then remove blank entries."""
    s = str(txt or '')
    return [item for raw in re.split(r'\n|(?=\[.*?\]\s*-)', s)
            if (item := _clean_action_item(raw))]

# ---- load the dashboard AI layer, with its own quality gate -----------------
# dash_ai_narrative keeps one row per generation run, so without a dedup here `nar`
# has multiple rows per page and .at[]/.loc[] silently return an arbitrary/duplicated
# row instead of the latest — keep only the most recently generated row per page.
nar = q("""
SELECT n.page, n.period_label, n.exec_summary_en, n.exec_summary_vi,
       n.narrative_en, n.narrative_vi, n.gen_status, n.generated_at,
       r.det_pass, r.verdict, r.judge_avg
FROM `{FQ}.dash_ai_narrative` n
LEFT JOIN (
  SELECT page, det_pass, verdict, judge_avg,
         ROW_NUMBER() OVER (PARTITION BY page ORDER BY reviewed_at DESC) rn
  FROM `{FQ}.dash_ai_review`
) r ON r.page = n.page AND r.rn = 1
QUALIFY ROW_NUMBER() OVER (PARTITION BY n.page ORDER BY n.generated_at DESC) = 1
""").set_index('page')

# det_traffic_claim_ok / det_causal_claim_ok are new gates. Without them here, a narrative
# rejected for an unsupported traffic claim or for saying spend "drove" revenue would show
# only the generic "failed review" message, with nothing to act on.
rev = q("""
SELECT page, det_numeric_ok, det_exec_summary_ok, det_consistency_ok,
       det_no_traffic_claim, det_traffic_claim_ok, det_causal_claim_ok,
       unsupported_numbers, missing_points, fabricated_points, verdict
FROM `{FQ}.dash_ai_review`
QUALIFY ROW_NUMBER() OVER (PARTITION BY page ORDER BY reviewed_at DESC) = 1
""")

# ---- review gate -----------------------------------------------------------
# A quarantined narrative failed the deterministic review, so it is NOT published.
# The report falls back to text built from dash_material_points, which is pure SQL
# and always safe. Blanking the section instead would leave the reader with nothing.
FAIL_REASON = {}
for p in nar.index:
    if str(nar.at[p, 'verdict']) != 'QUARANTINE':
        continue
    why = []
    rv = rev[rev.page == p].head(1)
    if len(rv):
        r0 = rv.iloc[0]
        if not r0.det_numeric_ok:
            why.append(f"số liệu không truy vết được về nguồn: {', '.join(r0.unsupported_numbers)}")
        if not r0.det_exec_summary_ok:
            why.append("tổng quan không nằm trong 3-5 câu")
        if not r0.det_consistency_ok and len(r0.missing_points):
            why.append(f"thiếu điểm trọng yếu: {', '.join(r0.missing_points)}")
        if len(r0.fabricated_points):
            why.append(f"mã điểm bịa đặt: {', '.join(r0.fabricated_points)}")
        if r0.det_no_traffic_claim is False:
            why.append("nhận định về lượt khách tại cửa hàng (footfall) — dữ liệu này không tồn tại")
        if r0.det_traffic_claim_ok is False:
            why.append("nhận định về lượt truy cập không có số liệu hỗ trợ trong khối dữ kiện")
        if r0.det_causal_claim_ok is False:
            why.append("quy kết doanh thu là do chi phí quảng cáo tạo ra — roas_channel chỉ là tỷ lệ hiệu quả")
    FAIL_REASON[p] = '; '.join(why) or 'không đạt vòng kiểm tra tự động'

for p, why in FAIL_REASON.items():
    banner(f'<b>{p}: nhận định AI bị chặn.</b> {why}. '
           f'Báo cáo hiển thị bản dựng từ SQL thay thế. Đặt SHOW_QUARANTINED = True để xem '
           f'bản AI không đạt, hoặc chạy lại CALL {FQ}.sp_dash_run_dashboard() để tạo lại.')

def ai_field(page, col):
    """AI text, but only if the page passed review (or override is on)."""
    if page not in nar.index:
        return None
    if page in FAIL_REASON and not SHOW_QUARANTINED:
        return None
    return scalar(nar.at[page, col])

# ---- deterministic fallback, built from the SQL point set -------------------
# Now carries point_type, so marketing points (M##) render as a spend/efficiency
# sentence rather than being described as a revenue move. Vietnamese throughout:
# an English fallback inside a Vietnamese report was the old failure mode.
pts = q("""
SELECT point_id, point_type, item_type, country, level_2,
       delta_nmv, pct_nmv, pct_bills, pct_aov,
       attainment_vs_full_month, expected_pace, pace_gap_pp,
       is_month_data_complete,
       spend, spend_p, pct_spend, roas_channel, roas_p, pct_roas,
       sessions, sessions_p, pct_sessions, ads_channels,
       material_reason
FROM `{FQ}.dash_material_points` ORDER BY point_type DESC, point_id
""")

def _pt_trade(r):
    move = 'tăng' if (r.delta_nmv or 0) >= 0 else 'giảm'
    bits = [f"{r.country} / {r.level_2}: doanh thu thuần {move} "
            f"{abs(r.pct_nmv or 0)*100:.{PCT_DP}f}% ({r.delta_nmv/1e6:,.0f} triệu VND)"]
    if pd.notna(r.pct_bills):
        bits.append(f"số hoá đơn {r.pct_bills*100:+.{PCT_DP}f}%")
    if pd.notna(r.pct_aov):
        bits.append(f"giá trị giỏ hàng {r.pct_aov*100:+.{PCT_DP}f}%")
    if pd.notna(r.pace_gap_pp) and r.is_month_data_complete is not False:
        bits.append(f"tiến độ {r.pace_gap_pp:+.{PCT_DP}f} điểm")
    s = ', '.join(bits)
    if r.is_month_data_complete is False:
        s += ' — dữ liệu tháng chưa đầy đủ, chưa đánh giá tiến độ'
    return s + f" ({r.point_id})"

def _pt_marketing(r):
    bits = [f"{r.country} / {r.level_2}: chi phí quảng cáo {(r.spend or 0)/1e6:,.0f} triệu VND"]
    if pd.notna(r.pct_spend):
        bits[0] += f" ({r.pct_spend*100:+.{PCT_DP}f}%)"
    if pd.notna(r.roas_channel):
        rr = f"hiệu quả trên chi phí {r.roas_channel:,.1f}"
        if pd.notna(r.pct_roas): rr += f" ({r.pct_roas*100:+.{PCT_DP}f}%)"
        bits.append(rr)
    if pd.notna(r.sessions) and (r.sessions or 0) > 0:
        ss = f"lượt truy cập {r.sessions:,.0f}"
        if pd.notna(r.pct_sessions): ss += f" ({r.pct_sessions*100:+.{PCT_DP}f}%)"
        bits.append(ss)
    if r.ads_channels:
        bits.append(f"nền tảng {r.ads_channels}")
    return ', '.join(bits) + f" ({r.point_id})"

def _pt_sentence(r):
    return _pt_marketing(r) if r.point_type == 'MARKETING' else _pt_trade(r)

def fallback_items(kind):
    d = pts[pts.item_type == kind]
    return [_pt_sentence(r) for _, r in d.iterrows()]

def fallback_summary():
    hi, lo = fallback_items('HIGHLIGHT'), fallback_items('LOWLIGHT')
    n_mkt = int((pts.point_type == 'MARKETING').sum())
    s = (f"Bản dựng từ SQL: {len(pts)} điểm trọng yếu trong kỳ — "
         f"{len(hi)} tích cực, {len(lo)} tiêu cực"
         + (f", trong đó {n_mkt} điểm về marketing. " if n_mkt else ". "))
    if hi: s += f"Điểm tích cực lớn nhất: {hi[0]}. "
    if lo: s += f"Điểm tiêu cực lớn nhất: {lo[0]}."
    return s

print('AI layer:', ', '.join(f'{p}={nar.at[p,"verdict"]}' for p in nar.index))
print(f'Điểm trọng yếu: {len(pts)} '
      f'({int((pts.point_type=="TRADE").sum())} kinh doanh, '
      f'{int((pts.point_type=="MARKETING").sum())} marketing)')

---
# 1. Tổng quan


In [ ]:
# Tổng quan điều hành từ `dash_ai_narrative` (tiếng Việt, nguyên văn). Sau đó là bảng KPI.
for page, label in [('PAGE1_WOW', 'So sánh tuần'),
                    ('PAGE2_MTD', 'Luỹ kế tháng so với chỉ tiêu')]:
    if page not in nar.index:
        continue

    meta = nar.loc[page]
    if isinstance(meta, pd.DataFrame):
        meta = meta.iloc[0]

    gen_at = pd.to_datetime(meta.generated_at).strftime('%Y-%m-%d') if pd.notna(meta.generated_at) else ''
    txt = scalar(ai_field(page, 'exec_summary_vi'))

    # meta.period_label / gen_at come from dash_ai_narrative, written whenever the AI job
    # last ran — NOT from WK_START/WK_END, which are recomputed fresh every run. If the AI
    # job hasn't rerun since before this report's WK_END, the header shows a stale period.
    stale = pd.notna(meta.generated_at) and pd.to_datetime(meta.generated_at).date() < WK_END

    if pd.notna(txt) and str(txt).strip():
        tag = ' (CHƯA KIỂM CHỨNG — không đạt vòng kiểm tra)' if page in FAIL_REASON else (' (DỮ LIỆU CŨ)' if stale else '')
        header = f'Tổng quan — {label} · {meta.period_label} · tạo ngày {gen_at}{tag}'
        color = ORANGE if (page in FAIL_REASON or stale) else BLUE

        # one sentence per paragraph, so a 5-sentence summary is readable
        cleaned_txt = re.sub(r'(?<=\.)\s+', '\n\n', str(txt).strip())
        prose(cleaned_txt, header, color)

        if stale:
            # MTD staleness reads as a day-of-month gap, not a week gap, so phrase it that way.
            expected = (f'ngày {WK_END.day} của {pd.Timestamp(WK_END).days_in_month} ngày'
                        if page == 'PAGE2_MTD' else f'tuần kết thúc {WK_END}')
            banner(f'<b>{label}: tổng quan AI đã cũ.</b> Được tạo ngày {gen_at} cho kỳ '
                   f'"{meta.period_label}", nhưng kỳ báo cáo hiện tại chạy tới {expected} '
                   f'(WK_END = {WK_END}). Job AI chưa chạy lại — hãy chạy '
                   f'<code>CALL {FQ}.sp_dash_run_dashboard()</code> để làm mới trước khi tin vào các mốc thời gian ở trên.')
    else:
        prose(fallback_summary(), f'Tổng quan — {label}', GREY, italic=True)

if not WEEK_COMPLETE:
    rows = '; '.join(f'{r.country} / {r.level_2} tới {r.complete_through}' for _, r in LAGGING.iterrows())
    extra = ''
    if len(WITHHELD):
        extra = (' Trong tháng, '
                 + '; '.join(f'{r.country} / {r.level_2} có {r.nmv_excluded_by_cutoff/1e6:,.0f} triệu VND '
                             f'đã tải nhưng bị giữ lại' for _, r in WITHHELD.iterrows())
                 + '.')
    banner(f'<b>Tuần báo cáo chưa đủ dữ liệu.</b> {rows}. Các kênh này bị thiếu hụt trong '
           f'{WK_START}–{WK_END}, và các số liệu WoW / MTD bên dưới thừa hưởng khoảng trống đó.{extra}')

In [ ]:
KPI_SQL = """
WITH fx AS (SELECT * FROM UNNEST([STRUCT('VIETNAM' AS country,'VND' AS ccy,1.0 AS r),
                                  ('THAILAND','THB',800.0),('CHINA','CNY',3800.0)])),
base AS (
  SELECT country, order_id, period_date,
         IF(IS_NAN(nmv) OR nmv IS NULL, 0, nmv) AS nmv,   -- NaN guard: SUM propagates NaN
         IF(IS_NAN(gmv) OR gmv IS NULL, 0, gmv) AS gmv,
         CASE WHEN level_1='Store' THEN 'bill'
              WHEN level_1='Wholesales' THEN '{ws}' ELSE 'order' END AS txn_class
  FROM `{FQ}.dash_base_order_day`
),
agg AS (
  SELECT country,'cur' AS pl, txn_class, order_id, nmv, gmv FROM base
    WHERE period_date BETWEEN '{ws_d}' AND '{we_d}'
  UNION ALL SELECT country,'prv',txn_class,order_id,nmv,gmv FROM base
    WHERE period_date BETWEEN '{ps_d}' AND '{pe_d}'
  -- >>> DAYS-ELAPSED CAPPING APPLIED HERE <<<
  -- 'mtd' and 'pm' are two EXPLICIT windows of the same length: days 1..N of this
  -- month and days 1..N of the prior month. The month boundary is no longer derived
  -- inline, so the current and prior sides cannot drift apart.
  UNION ALL SELECT country,'mtd',txn_class,order_id,nmv,gmv FROM base
    WHERE period_date BETWEEN '{mtd_s}' AND '{mtd_e}'
  UNION ALL SELECT country,'pm',txn_class,order_id,nmv,gmv FROM base
    WHERE period_date BETWEEN '{pm_s}' AND '{pm_e}'
)
SELECT a.country, f.ccy AS currency_code, a.pl AS period_label,
  COUNT(DISTINCT IF(txn_class='bill',  order_id, NULL)) AS bills,
  COUNT(DISTINCT IF(txn_class='order', order_id, NULL)) AS orders,
  COUNT(DISTINCT order_id) AS transactions,
  SUM(gmv) AS gmv_vnd, SUM(nmv) AS nmv_vnd,
  SAFE_DIVIDE(SUM(nmv), COUNT(DISTINCT order_id)) AS aov_vnd
FROM agg a JOIN fx f ON f.country = a.country
GROUP BY a.country, f.ccy, a.pl
"""

k = q(KPI_SQL, ws=WHOLESALE_CLASS, ws_d=WK_START, we_d=WK_END, ps_d=PW_START, pe_d=PW_END,
      mtd_s=MTD_START, mtd_e=MTD_END, pm_s=PM_START, pm_e=PM_END)

# FIXED. month_target_revenue is the whole-month total for a level_2, repeated on every
# date row. Summing it across a date range multiplied the target by the number of days:
# measured 705.0bn instead of 23.5bn for August, a 30x overstatement that pushed MTD
# attainment from ~104% down to ~3%. Read one value per level_2 with MAX(), then sum the
# level_2 values - the same guard the run-rate table in section 6 already had.
tgt = q("""
SELECT country, SUM(t) AS mtd_target_vnd FROM (
  SELECT country, level_2, MAX(month_target_revenue) AS t
  FROM `{FQ}.dash_target_day`
  -- same month as the MTD actuals above, read from the SAME constant rather than
  -- re-derived inline, so the target and the actuals cannot drift apart.
  -- This is a single-period lookup (full-month target); it has no prior-period side
  -- and is deliberately NOT days-elapsed capped - see the note under the table.
  WHERE DATE_TRUNC(sale_date, MONTH) = DATE '{mtd_s}'
    AND NOT is_unmapped_channel
  GROUP BY country, level_2
) GROUP BY country
""", we_d=WK_END, mtd_s=MTD_START).set_index('country')

cur = k[k.period_label=='cur'].set_index('country')
prv = k[k.period_label=='prv'].set_index('country')
mtd = k[k.period_label=='mtd'].set_index('country')
pm  = k[k.period_label=='pm' ].set_index('country')   # prior month, days 1..N

rows = []
for c in cur.index:
    ccy, rate = FX.get(c, ('VND', 1.0))
    show_local = (c != 'VIETNAM')   # VN local IS the reporting currency; do not duplicate
    for metric, col, is_money in [('Số hoá đơn (offline)','bills',False),
                                  ('Số đơn (online)','orders',False),
                                  ('GMV','gmv_vnd',True), ('NMV','nmv_vnd',True),
                                  ('Giá trị giỏ hàng (AOV)','aov_vnd',True)]:
        cv = cur.at[c,col]
        pv = prv.at[c,col] if c in prv.index else None
        wow = (cv/pv - 1)*100 if (pv not in (None,0) and pd.notna(pv)) else None
        # MoM on the capped windows: days 1..N this month vs days 1..N last month
        mv = mtd.at[c,col] if c in mtd.index else None
        qv = pm.at[c,col]  if c in pm.index  else None
        mom = (mv/qv - 1)*100 if (None not in (mv,qv) and qv not in (0,)
                                  and pd.notna(mv) and pd.notna(qv)) else None
        if metric == 'NMV':
            t = tgt.at[c,'mtd_target_vnd'] if c in tgt.index else None
            ach = f'{mtd.at[c,"nmv_vnd"]/t*100:.{PCT_DP}f}%' if (t and pd.notna(t)) else 'chưa có chỉ tiêu'
        else:
            ach = '—'
        rows.append({'country':c, 'metric':metric,
          'value_local': (f'{cv/rate:,.{MONEY_DP}f}' if is_money else f'{cv:,.0f}') if show_local else '—',
          'value_vnd':   f'{cv:,.{MONEY_DP}f}' if is_money else f'{cv:,.0f}',
          'currency_code': ccy,
          'wow_pct': f'{wow:+.{PCT_DP}f}%' if wow is not None else '—',
          'mom_pct': f'{mom:+.{PCT_DP}f}%' if mom is not None else '—',
          'mtd_vs_target': ach})

# Group total — VND only. Local is em dash: never sum mixed currencies.
for metric, col, is_money in [('Số hoá đơn (offline)','bills',False),('Số đơn (online)','orders',False),
                              ('GMV','gmv_vnd',True),('NMV','nmv_vnd',True)]:
    cv, pv = cur[col].sum(), prv[col].sum()
    wow = (cv/pv - 1)*100 if pv else None
    ach = (f'{mtd.nmv_vnd.sum()/tgt.mtd_target_vnd.sum()*100:.{PCT_DP}f}%'
           if (metric=='NMV' and tgt.mtd_target_vnd.sum()) else '—')
    mv, qv = mtd[col].sum(), pm[col].sum() if len(pm) else 0
    mom = (mv/qv - 1)*100 if qv else None
    rows.append({'country':'TẤT CẢ (chỉ VND)','metric':metric,'value_local':'—',
      'value_vnd':f'{cv:,.{MONEY_DP}f}','currency_code':'VND',
      'wow_pct':f'{wow:+.{PCT_DP}f}%' if wow is not None else '—',
      'mom_pct':f'{mom:+.{PCT_DP}f}%' if mom is not None else '—','mtd_vs_target':ach})

tc, tp = cur.transactions.sum(), prv.transactions.sum()
aov_c = cur.nmv_vnd.sum()/tc if tc else None
aov_p = prv.nmv_vnd.sum()/tp if tp else None
rows.append({'country':'TẤT CẢ (chỉ VND)','metric':'Giá trị giỏ hàng (AOV)','value_local':'—',
  'value_vnd':f'{aov_c:,.{MONEY_DP}f}' if aov_c else '—','currency_code':'VND',
  'wow_pct':f'{(aov_c/aov_p-1)*100:+.{PCT_DP}f}%' if (aov_c and aov_p) else '—',
  'mom_pct':'—', 'mtd_vs_target':'—'})

head(f'KPI — tuần {WK_START} tới {WK_END}',
     f'hoá đơn = offline, đơn = online, AOV = NMV / (hoá đơn + đơn). '
     f'MoM so sánh cùng kỳ: {MTD_START}–{MTD_END} với {PM_START}–{PM_END} ({N_DAYS} ngày mỗi bên). '
     f'MTD so với chỉ tiêu tính trên chỉ tiêu cả tháng.')

df_kpi = pd.DataFrame(rows).rename(columns={
    'country': 'Thị trường',
    'metric': 'Chỉ số',
    'value_local': 'Giá trị (nội tệ)',
    'value_vnd': 'Giá trị (VND)',
    'currency_code': 'Mệnh giá',
    'wow_pct': 'Tăng trưởng so với tuần trước',
    'mom_pct': f'So với cùng kỳ tháng trước ({N_DAYS} ngày)',
    'mtd_vs_target': 'MTD so với chỉ tiêu'
})

def highlight_wow(val):
    if not isinstance(val, str) or val in ('—', '', 'chưa có chỉ tiêu'):
        return ''
    try:
        num = float(val.replace('%', '').replace('+', ''))
        if num < 0:   return f'color: {RED}; font-weight: bold;'
        elif num > 0: return f'color: {GREEN}; font-weight: bold;'
    except ValueError:
        pass
    return ''

def highlight_attain(val):
    if not isinstance(val, str) or val in ('—','','chưa có chỉ tiêu'):
        return ''
    try:
        num = float(val.replace('%',''))
        return (f'color: {GREEN}; font-weight: bold;' if num >= 100
                else f'color: {RED}; font-weight: bold;' if num < 80 else '')
    except ValueError:
        return ''

display(df_kpi.style.hide(axis='index')
        .map(highlight_wow,    subset=['Tăng trưởng so với tuần trước',
                                       f'So với cùng kỳ tháng trước ({N_DAYS} ngày)'])
        .map(highlight_attain, subset=['MTD so với chỉ tiêu'])
        .set_table_styles(TH))

---
# 2. Highlights


In [ ]:
# Từ `dash_ai_narrative.narrative_vi`, nguyên văn. Mỗi mục kết thúc bằng mã điểm
# (`P01`, `P02`, … cho kinh doanh; `M01`, `M02`, … cho marketing) mà nó gắn vào, nên
# truy vết được về đúng tập điểm đã lọc trong SQL.
for page, label in [('PAGE1_WOW', 'So sánh tuần'),
                    ('PAGE2_MTD', 'Luỹ kế tháng so với chỉ tiêu')]:
    if page not in nar.index:
        continue

    hi_items, _, _ = split_narrative(scalar(ai_field(page, 'narrative_vi')))
    src = 'AI' if hi_items else 'Dữ liệu dựng sẵn'
    if not hi_items:
        hi_items = fallback_items('HIGHLIGHT')

    head(f'{label} — {len(hi_items)} điểm tích cực', f'nguồn: {src}')
    bullets([f'<b style="color:{BLUE}">Điểm tích cực —</b> {t}' for t in hi_items],
            BLUE, 'điểm tích cực')

---
# 3. Lowlights


In [ ]:
for page, label in [('PAGE1_WOW', 'So sánh tuần'),
                    ('PAGE2_MTD', 'Luỹ kế tháng so với chỉ tiêu')]:
    if page not in nar.index:
        continue

    _, lo, _ = split_narrative(scalar(ai_field(page, 'narrative_vi')))
    src = 'AI' if lo else 'Dữ liệu dựng sẵn'
    if not lo:
        lo = fallback_items('LOWLIGHT')

    head(f'{label} — {len(lo)} điểm cần lưu ý', f'nguồn: {src}')
    bullets([f'<b style="color:{ORANGE}">Điểm cần lưu ý —</b> {t}' for t in lo],
            ORANGE, 'điểm cần lưu ý')

---
# 4. Marketing — quảng cáo & lượt truy cập

In [ ]:

# Nguồn: `dash_trend_marketing_weekly` — chi phí quảng cáo (Meta / TikTok / Google Ads) và
# lượt truy cập website từ GA4, gộp lên level_2.
#
# ĐỌC ĐÚNG ba con số này:
#  1. `roas_channel` = NMV / chi phí của kênh. Đây là **tỷ lệ hiệu quả**, KHÔNG phải bằng chứng
#     chi phí tạo ra doanh thu — nguồn quảng cáo quy kết theo kênh và theo ngày.
#  2. `delivery` = COALESCE(impressions, reach). Google báo impressions, Meta/TikTok báo reach;
#     đây là hai đại lượng khác nhau nên không cộng chung.
#  3. Lượt truy cập chỉ có cho **website**. Cửa hàng vật lý không đo được lượt khách (footfall).
#  4. MỚI 2026-09-06 — `cpc` chia cho **chi phí đấu giá**, không phải tổng chi phí. TikTok
#     GMV Max tiêu tiền nhưng không báo lượt nhấp/hiển thị, nên gộp chung làm CPC và CPM sai
#     lệch tới 31 lần (Tiktok Thailand: 46.885 đ so với 1.522 đ thực). Cột "Chi phí đấu giá"
#     và "Chi phí GMV Max" tách rõ hai loại.
#  5. `platform_orders` / doanh thu nền tảng là số TikTok Shop TỰ quy kết — không phải đơn
#     hàng trong hệ thống LSOUL, và không được cộng với `nmv`.

mkt = q("""
SELECT country, level_2,
       STRING_AGG(DISTINCT ads_channels, '+')            AS ads_channels,
       SUM(spend)                                        AS spend,
       SUM(delivery)                                     AS delivery,
       SUM(clicks)                                       AS clicks,
       SUM(sessions)                                     AS sessions,
       SUM(nmv)                                          AS nmv,
       SUM(bills)                                        AS bills,
       SUM(spend_prev_wk)                                AS spend_prev,
       SAFE_DIVIDE(SUM(nmv), NULLIF(SUM(spend),0))       AS roas_channel,
       SAFE_DIVIDE(SUM(spend), NULLIF(SUM(nmv),0))       AS cost_of_sale,
       SUM(spend_auction)                                AS spend_auction,
       SUM(spend_gmv_max)                                AS spend_gmv_max,
       SUM(platform_orders)                              AS platform_orders,
       SUM(platform_gross_revenue_vnd)                   AS platform_revenue,
       -- CPC phải chia cho chi phí ĐẤU GIÁ, không phải tổng chi phí: TikTok GMV Max có
       -- chi phí nhưng KHÔNG báo lượt nhấp / hiển thị. Lấy tổng chi phí chia cho lượt nhấp
       -- chỉ của đấu giá là sai — đo được trên Tiktok Thailand: 46.885 đ so với 1.522 đ thực.
       SAFE_DIVIDE(SUM(spend_auction), NULLIF(SUM(clicks),0)) AS cpc,
       SAFE_DIVIDE(SUM(paid_sessions), NULLIF(SUM(sessions),0)) AS paid_share
FROM `{FQ}.dash_trend_marketing_weekly`
WHERE week_start = DATE '{ws_d}' AND (spend > 0 OR sessions > 0)
GROUP BY country, level_2
ORDER BY spend DESC
""", ws_d=WK_START)

# BigQuery NUMERIC/BIGNUMERIC columns (spend, nmv, ...) come back as decimal.Decimal,
# which cannot be divided by a plain float (1e6) below — cast the summed money/count
# columns to float64 right after the query so all downstream arithmetic is consistent.
_NUMERIC_COLS = ['spend', 'delivery', 'clicks', 'sessions', 'nmv', 'bills', 'spend_prev',
                 'roas_channel', 'cost_of_sale', 'cpc', 'paid_share',
                 'spend_auction', 'spend_gmv_max', 'platform_orders', 'platform_revenue']
mkt[_NUMERIC_COLS] = mkt[_NUMERIC_COLS].astype('float64')

if not len(mkt):
    head('Marketing', 'không ghi nhận chi phí quảng cáo hoặc lượt truy cập trong tuần')
else:
    mkt['wow'] = mkt.apply(
        lambda r: (r.spend/r.spend_prev - 1)*100 if (r.spend_prev and pd.notna(r.spend_prev)) else None, axis=1)

    out = mkt.assign(
        Kênh          = mkt.country + ' / ' + mkt.level_2,
        NềnTảng       = mkt.ads_channels.fillna('—'),
        ChiPhí        = mkt.spend.map(lambda v: f'{v:,.0f}'),
        ChiPhíWoW     = mkt.wow.map(lambda v: '—' if v is None or pd.isna(v) else f'{v:+.{PCT_DP}f}%'),
        HiểnThị       = mkt.delivery.map(lambda v: '—' if pd.isna(v) else f'{v:,.0f}'),
        LượtNhấp      = mkt.clicks.map(lambda v: '—' if pd.isna(v) else f'{v:,.0f}'),
        CPC           = mkt.cpc.map(lambda v: '—' if pd.isna(v) else f'{v:,.0f}'),
        ChiPhíĐấuGiá  = mkt.spend_auction.map(lambda v: '—' if pd.isna(v) else f'{v:,.0f}'),
        ChiPhíGMVMax  = mkt.spend_gmv_max.map(lambda v: '—' if pd.isna(v) or v == 0 else f'{v:,.0f}'),
        LượtTruyCập   = mkt.sessions.map(lambda v: '—' if pd.isna(v) or v == 0 else f'{v:,.0f}'),
        DoanhThu      = mkt.nmv.map(lambda v: f'{v:,.0f}'),
        HiệuQuả       = mkt.roas_channel.map(lambda v: '—' if pd.isna(v) else f'{v:,.1f}'),
        TỷLệChiPhí    = mkt.cost_of_sale.map(lambda v: '—' if pd.isna(v) else f'{v*100:.{PCT_DP}f}%'),
    )[['Kênh','NềnTảng','ChiPhí','ChiPhíĐấuGiá','ChiPhíGMVMax','ChiPhíWoW','HiểnThị',
       'LượtNhấp','CPC','LượtTruyCập','DoanhThu','HiệuQuả','TỷLệChiPhí']].rename(columns={
        'ChiPhíĐấuGiá':'Chi phí đấu giá (VND)', 'ChiPhíGMVMax':'Chi phí GMV Max (VND)',
        'NềnTảng':'Nền tảng', 'ChiPhí':'Chi phí (VND)', 'ChiPhíWoW':'Chi phí so tuần trước',
        'HiểnThị':'Hiển thị / tiếp cận', 'LượtNhấp':'Lượt nhấp',
        'LượtTruyCập':'Lượt truy cập website', 'DoanhThu':'Doanh thu thuần (VND)',
        'HiệuQuả':'Doanh thu / chi phí', 'TỷLệChiPhí':'Tỷ lệ chi phí trên doanh thu'})

    head(f'Quảng cáo & lượt truy cập — tuần {WK_START} tới {WK_END}',
         'doanh thu / chi phí là tỷ lệ hiệu quả của kênh, không phải mức đóng góp nhân quả')
    table(out)

    tot_spend, tot_nmv = mkt.spend.sum(), mkt.nmv.sum()
    zero_rev = mkt[(mkt.spend > 0) & (mkt.nmv <= 0)]
    if len(zero_rev):
        banner('<b>Chi phí không khớp doanh thu.</b> '
               + '; '.join(f'{r.country} / {r.level_2} chi {r.spend/1e6:,.1f} triệu VND '
                           f'nhưng không có dòng doanh thu nào khớp' for _, r in zero_rev.iterrows())
               + f'. Tổng {zero_rev.spend.sum()/1e6:,.1f} triệu VND'
               + (f' ({zero_rev.spend.sum()/tot_spend*100:.{PCT_DP}f}% chi phí tuần)' if tot_spend else '')
               + '. Nguyên nhân là ánh xạ level_3 của nguồn quảng cáo, không phải lỗi tính toán.')

    cos = f'{tot_spend/tot_nmv*100:.{PCT_DP}f}%' if tot_nmv else '—'
    banner(f'Tổng chi phí quảng cáo tuần: <b>{tot_spend/1e6:,.1f} triệu VND</b> trên '
           f'<b>{tot_nmv/1e6:,.0f} triệu VND</b> doanh thu thuần của các kênh có quảng cáo — '
           f'tỷ lệ chi phí trên doanh thu <b>{cos}</b>.', BLUE)

# Nhận định marketing (M##) — lấy từ narrative AI nếu có, nếu không thì dựng từ SQL.
mkt_pts = pts[pts.point_type == 'MARKETING']
if len(mkt_pts):
    ai_mkt = []
    for page in ('PAGE1_WOW', 'PAGE2_MTD'):
        if page not in nar.index:
            continue
        hi, lo, _ = split_narrative(scalar(ai_field(page, 'narrative_vi')))
        ai_mkt += [t for t in (hi + lo) if re.search(r'\bM[0-9]{2}\b', t)]

    src = 'AI' if ai_mkt else 'dựng từ SQL (AI bị chặn hoặc không đề cập)'
    items = ai_mkt or [_pt_sentence(r) for _, r in mkt_pts.iterrows()]
    head(f'Nhận định marketing — {len(items)} điểm trọng yếu', f'nguồn: {src}')
    bullets([f'<b style="color:{ORANGE}">Marketing —</b> {t}' for t in items],
            ORANGE, 'nhận định marketing')
else:
    head('Nhận định marketing',
         'không có biến động nào vượt ngưỡng trọng yếu về chi phí hoặc hiệu quả trong tuần')


In [ ]:
# ============ PHÂN RÃ NGUYÊN NHÂN: bao nhiêu phần do marketing? ============
# Nguồn: `dash_marketing_drivers` (mới 2026-09-06).
#
# CÁCH ĐỌC — hai tầng, cộng khớp tuyệt đối:
#   Tầng 1  Doanh thu = Ngân sách x Hiệu quả (ROAS). Phần giao thoa chia đôi, nên
#           "do ngân sách" + "do hiệu quả" = đúng bằng mức biến động doanh thu.
#   Tầng 2  Tách tiếp phần "do hiệu quả" theo tỷ trọng logarit (LMDI):
#           - kênh ĐẤU GIÁ  : ROAS = Tỷ lệ chuyển đổi x Giá trị đơn / Chi phí mỗi lượt nhấp
#           - kênh GMV MAX  : ROAS = Giá trị đơn / Chi phí mỗi đơn
#             (GMV Max không báo lượt nhấp nên không thể tách theo lượt nhấp)
#
# QUAN TRỌNG — "doanh thu" ở đây là NMV của LSOUL, KHÔNG phải doanh thu quy kết quảng cáo.
# Meta không báo đơn hàng hay doanh thu nào cả (32% chi phí), nên một phân rã dựa trên quy
# kết nền tảng sẽ rỗng ở phần lớn kênh. Vì vậy bảng này trả lời "doanh thu kênh biến động
# thế nào so với ngân sách và hiệu quả", KHÔNG phải "quảng cáo tạo ra bao nhiêu doanh thu".
#
# Các dòng `driver_confidence` khác 'ok' bị loại: kênh không có chi phí ở một trong hai kỳ,
# hoặc thiếu thành phần để tách tầng 2.

_VI = {'spend':'Ngân sách', 'roas':'Hiệu quả', 'cvr':'Tỷ lệ chuyển đổi',
       'aov':'Giá trị đơn', 'cpc':'Chi phí mỗi lượt nhấp', 'cpo':'Chi phí mỗi đơn'}
_PERIOD_VI = {'WTD':'Tuần đến hiện tại', 'MTD':'Tháng đến hiện tại'}

try:
    drv = q("""
    SELECT period_type, period_start_date, period_end_date, prv_start_date, prv_end_date,
           country, level_2, level_3, driver_level, driver_name, driver_basis,
           driver_confidence, level1_status,
           contribution_vnd, revenue_cur, revenue_prv, revenue_delta, revenue_pct,
           delta_from_spend, delta_from_roas, spend_cur, spend_prv, roas_cur, roas_prv
    FROM `{FQ}.dash_marketing_drivers`
    WHERE level1_status = 'ok'
    """)
except Exception as _e:
    drv = pd.DataFrame()
    head('Phân rã nguyên nhân marketing',
         'bảng `dash_marketing_drivers` chưa được triển khai — bỏ qua phần này')

if len(drv):
    for _c in ['contribution_vnd','revenue_cur','revenue_prv','revenue_delta','revenue_pct',
               'delta_from_spend','delta_from_roas','spend_cur','spend_prv','roas_cur','roas_prv']:
        drv[_c] = drv[_c].astype('float64')
    drv['Kênh'] = drv.country + ' / ' + drv.level_2 + ' — ' + drv.level_3.fillna('—')

    for ptype in ('WTD', 'MTD'):
        d = drv[drv.period_type == ptype]
        if not len(d):
            continue
        _w = d.iloc[0]
        head(f'Phân rã nguyên nhân — {_PERIOD_VI[ptype]}',
             f'{_w.period_start_date} tới {_w.period_end_date} so với '
             f'{_w.prv_start_date} tới {_w.prv_end_date} (cùng số ngày) · '
             f'doanh thu = NMV của LSOUL, không phải doanh thu quy kết quảng cáo')

        base = (d[d.driver_level == 1]
                .drop_duplicates('Kênh')
                .set_index('Kênh')[['revenue_prv','revenue_cur','revenue_delta','revenue_pct',
                                    'delta_from_spend','delta_from_roas','spend_prv','spend_cur',
                                    'roas_prv','roas_cur','driver_basis']])
        lev = (d[(d.driver_level == 2) & (d.driver_confidence == 'ok')]
               .pivot_table(index='Kênh', columns='driver_name',
                            values='contribution_vnd', aggfunc='sum'))
        tbl = base.join(lev, how='left').sort_values('revenue_delta', key=abs, ascending=False)

        _m = lambda v: '—' if pd.isna(v) else f'{v/1e6:,.0f}'
        out = pd.DataFrame({
            'Kênh': tbl.index,
            'Doanh thu kỳ trước (tr)': tbl.revenue_prv.map(_m),
            'Doanh thu kỳ này (tr)':   tbl.revenue_cur.map(_m),
            'Biến động (tr)':          tbl.revenue_delta.map(_m),
            '%':                       tbl.revenue_pct.map(lambda v: '—' if pd.isna(v) else f'{v*100:+.{PCT_DP}f}%'),
            'Do ngân sách (tr)':       tbl.delta_from_spend.map(_m),
            'Do hiệu quả (tr)':        tbl.delta_from_roas.map(_m),
        })
        for k in ('cvr','aov','cpc','cpo'):
            if k in tbl.columns:
                out[_VI[k] + ' (tr)'] = tbl[k].map(_m).values
        table(out)

        # Đòn bẩy lớn nhất của kênh biến động mạnh nhất
        top = tbl.iloc[0]
        levers = {k: top[k] for k in ('cvr','aov','cpc','cpo')
                  if k in tbl.columns and pd.notna(top[k])}
        if levers:
            k = max(levers, key=lambda x: abs(levers[x]))
            banner(f'<b>{tbl.index[0]}</b>: doanh thu biến động '
                   f'<b>{top.revenue_delta/1e6:,.0f} triệu VND</b> — '
                   f'{top.delta_from_spend/1e6:,.0f} do ngân sách, '
                   f'{top.delta_from_roas/1e6:,.0f} do hiệu quả. '
                   f'Trong phần hiệu quả, đòn bẩy lớn nhất là <b>{_VI[k].lower()}</b> '
                   f'({levers[k]/1e6:,.0f} triệu VND).', BLUE)

    _skipped = drv[drv.driver_confidence != 'ok'].drop_duplicates(['period_type','Kênh'])
    if len(_skipped):
        banner(f'{len(_skipped)} kênh-kỳ không tách được tầng 2 '
               f'(' + ', '.join(sorted(set(_skipped.driver_confidence))) + '). '
               f'Thường là kênh không có lượt nhấp hoặc không có đơn hàng ở một trong hai kỳ.',
               ORANGE)


---
# 5. Hàng hoá (Stock analysis)

Nguồn: `report_stats_product` ở cấp **variation_code** (một dòng mỗi SKU), nối với doanh thu tuần thật từ `dash_base_sku_day.revenue_vn`.

In [ ]:
# STOCK action items. Từ 2026-09-06, phần chữ tiếng Việt được **gemini-3.5-flash viết
# trực tiếp trong `dash_action_items`** (cột `action_item_vi`), không còn phụ thuộc vào bản
# dịch trong narrative. Mọi CON SỐ vẫn do SQL tính; model chỉ viết câu. Năm chốt kiểm tra
# trong bảng đó (số liệu, mã SKU, câu nhân quả, dấu tiếng Việt, độ dài) sẽ tự động lùi về
# câu SQL nếu model sai — `vi_source` cho biết đang đọc bản nào.
#
# ĐÃ SỬA: ô này trước đây lọc section='PERFORMANCE', nên mục Hàng hoá hiển thị các hành động
# về hiệu suất, còn 5 mục STOCK thì không xuất hiện ở bất kỳ đâu trong báo cáo. STOCK items
# còn mang page='BOTH', nên bộ lọc theo page cũng sẽ không khớp.

stock_acts = q("""
SELECT action_no, owner, action_item, action_item_vi, source_item, impact_vnd,
       text_source, vi_source, gen_status
FROM `{FQ}.dash_action_items`
WHERE section = 'STOCK'
ORDER BY action_no
""")

# PERFORMANCE actions are used in section 6; loaded once here.
perf = q("""
SELECT page, action_no, owner, action_item, action_item_vi, source_item, impact_vnd,
       text_source, vi_source, gen_status
FROM `{FQ}.dash_action_items`
WHERE section='PERFORMANCE'
ORDER BY page, action_no
""")

# Bao nhiêu dòng thực sự là tiếng Việt do model viết, bao nhiêu dòng đã lùi về câu SQL.
_all_acts = pd.concat([stock_acts, perf], ignore_index=True)
VI_OK   = int((_all_acts.vi_source == 'ai_vi').sum())
VI_BACK = int((_all_acts.vi_source != 'ai_vi').sum())

if len(stock_acts):
    _n = int((stock_acts.vi_source == 'ai_vi').sum())
    head('Hành động về hàng hoá',
         f'số liệu được tính toán sẵn, văn bản tạo bởi AI'
         f'({_n}/{len(stock_acts)} dòng qua đủ 5 mục kiểm tra độ chính xác của AI)')
    stock_items = []
    for _, r in stock_acts.iterrows():
        for action in split_actions(r.action_item_vi or r.action_item):
            impact = (f' · tác động ước tính {r.impact_vnd/1e6:,.0f} triệu VND'
                      if pd.notna(r.impact_vnd) else '')
            stock_items.append(
                f'{action}'
                f'<div style="font:400 11.5px system-ui;color:{GREY};margin-top:5px">'
                f'Dữ liệu từ {r.source_item}{impact}</div>')
    bullets(stock_items, ORANGE, 'hành động về hàng hoá')
else:
    head('Hành động về hàng hoá')
    bullets([], ORANGE, 'hành động về hàng hoá')

In [ ]:
# ============================ PHÂN TÍCH HÀNG HOÁ =============================
# Nguồn duy nhất: `report_stats_product` ở đúng cấp SKU (variation_code) — bảng này
# đã là MỘT DÒNG cho mỗi variation_code tại một mốc snapshot (is_latest = 1), nên
# không cần gộp lại. Doanh thu tuần lấy từ `dash_base_sku_day.revenue_vn`.
#
# BA LỖI ĐÃ SỬA Ở PHẦN NÀY
#
# 1. DOANH THU THEO SKU TRƯỚC ĐÂY LÀ SỐ BỊA. Ô cũ tính
#       nmv_week_vnd = units_week * (tổng NMV toàn hệ / tổng số lượng toàn hệ)
#    tức nhân số lượng với MỘT đơn giá bình quân duy nhất (2.724k VND) cho mọi SKU —
#    một chiếc váy và một chiếc khăn được định giá như nhau. Đo trên tuần 24–30/08:
#    sai số tuyệt đối trung bình 81% mỗi SKU, và 237/628 SKU lệch hơn 50%.
#    Vì giá trị đó = số lượng × hằng số, xếp hạng theo "NMV" chỉ là xếp hạng theo số
#    lượng, nên chú thích "ties broken by NMV" ở bảng cũ không có tác dụng gì.
#    `dash_base_sku_day` vốn đã có `revenue_vn` thật theo từng SKU — nay dùng cột đó.
#
# 2. GỘP SAI CẤP. Ô cũ dùng REGEXP_EXTRACT(variation_code, r'^([^-]+)') để cắt mã ở
#    dấu gạch ngang đầu tiên, gộp các biến thể size/màu lại. Cách đó gần trùng với cột
#    `product_code` nhưng lệch ở 41/4.393 dòng. Nay nối trực tiếp theo `variation_code`
#    — khớp 628/628 SKU của tuần, không sót dòng nào.
#
# 3. THIẾU BỘ CHỌN DÒNG HIỆN HÀNH. Ô cũ đọc `report_stats_product` mà không lọc gì cả.
#    Nay lọc `is_latest = 1` — đây là bộ chọn dòng hiện hành đúng và duy nhất.
#    KHÔNG lọc theo `period_date`: cột đó là NGÀY CẬP NHẬT GẦN NHẤT CỦA TỪNG SKU, không
#    phải mốc snapshot chung, và không mô tả một khoảng thời gian nào. Lọc theo
#    MAX(period_date) sẽ âm thầm loại bỏ mọi SKU không được cập nhật đúng ngày hôm đó.
#    Phần này vì vậy cũng không hiển thị period_date như thể là ngày chốt số liệu.
#
# BA CỘT KHÔNG DÙNG ĐƯỢC, ĐÃ TRÁNH
#   `overstock_flag`          : luôn FALSE cho cả 4.393 dòng, trong khi `alert_code`
#                               gắn cờ 776 SKU là OVERSTOCK_*. Dùng alert_code.
#   `sales_trend_direction`   : chỉ có 2 giá trị, và 'STABLE' trùng khít với nhóm
#                               stockout_risk_level = 'NO_SALES' (2.548 dòng cả hai).
#                               'STABLE' ở đây nghĩa là KHÔNG BÁN ĐƯỢC, không phải ổn
#                               định — nên không hiển thị như một tín hiệu tốt.
#   `recommended_discount_pct`: bằng 0 ở 1.368/1.534 SKU tồn kho dư, cao nhất chỉ 18%.
#                               Dùng `suggested_discount_pct` — cột này có giá trị ở cả
#                               1.534 SKU, theo bậc 30/40/50%, tức là dùng được thật.

STOCK_SQL = """
WITH wk AS (   -- doanh thu và số lượng THẬT theo từng SKU trong tuần báo cáo
  SELECT variation_code,
         SUM(units_sold)                        AS units_week,
         SUM(revenue_vn)                        AS revenue_week_vnd,
         SUM(total_orders)                      AS orders_week
  FROM `{FQ}.dash_base_sku_day`
  WHERE period_date BETWEEN '{ws_d}' AND '{we_d}'
  GROUP BY 1
),
sp AS (        -- một dòng mỗi variation_code; lọc snapshot rõ ràng
  SELECT
     variation_code, product_code, product_name, variation_attributes
    ,CAST(retail_price AS FLOAT64)                              AS retail_price
    ,product_days_since_launch, days_range, is_inactive, is_sale_end
    ,CAST(sl_ton_kho AS FLOAT64)                                AS stock_units
    ,CAST(tong_gia_tri_hang_ton_kho_theo_gia_ban AS FLOAT64)    AS stock_value_vnd
    ,CAST(units_sold_day_range AS FLOAT64)                      AS units_in_window
    ,CAST(revenue AS FLOAT64)                                   AS revenue_window_vnd
    ,CAST(sell_through_rate AS FLOAT64)                         AS sell_through
    ,CAST(sale_velocity AS FLOAT64)                             AS sale_velocity
    ,CAST(expected_velocity AS FLOAT64)                         AS expected_velocity
    ,CAST(DOH AS FLOAT64)                                       AS doh
    ,doh_is_infinite, health_score, colour_bucket, alert_code, alert_severity
    ,overstock_risk_score
    ,CAST(excess_units AS FLOAT64)                              AS excess_units
    ,CAST(excess_value AS FLOAT64)                              AS excess_value_vnd
    ,CAST(suggested_discount_pct   AS FLOAT64)                  AS suggested_discount_pct
    ,CAST(recommended_discount_pct AS FLOAT64)                  AS rec_discount_pct
    ,CAST(days_until_stockout AS FLOAT64)                       AS days_until_stockout
    ,stockout_risk_level
    ,CAST(restock_urgency_score AS FLOAT64)                     AS restock_urgency
    ,CAST(weeks_of_cover AS FLOAT64)                            AS weeks_of_cover
    ,CAST(recommended_restock_qty AS FLOAT64)                   AS rec_restock_qty
    ,CAST(forecast_units_next_30d AS FLOAT64)                   AS forecast_30d
    ,CAST(sku_units_last_30d AS FLOAT64)                        AS units_30d
    ,CAST(avg_daily_sales_7d AS FLOAT64)                        AS avg_daily_7d
    ,CAST(avg_daily_sales_30d AS FLOAT64)                       AS avg_daily_30d
    ,is_top_seller, is_all_time_top_10, is_top_20_30d, top_seller_categories
    ,ai_recommendation_note
  FROM `{FQ}.report_stats_product`
  WHERE is_latest = 1          -- bo chon dong hien hanh DUY NHAT; xem ghi chu 3
)
SELECT sp.*,
       IFNULL(wk.units_week, 0)       AS units_week,
       IFNULL(wk.revenue_week_vnd, 0) AS revenue_week_vnd,
       IFNULL(wk.orders_week, 0)      AS orders_week
FROM sp
LEFT JOIN wk USING (variation_code)   -- khớp trực tiếp, không cần regex
WHERE product_name is not null
"""
sk = q(STOCK_SQL, ws_d=WK_START, we_d=WK_END)

ALERT_VI = {
    'HEALTHY':            'Khoẻ mạnh',
    'DEAD_STOCK':         'Hàng chết',
    'OVERSTOCK_CRITICAL': 'Tồn kho nghiêm trọng',
    'OVERSTOCK_HIGH':     'Tồn kho cao',
    'OVERSTOCK_WATCH':    'Tồn kho cần theo dõi',
    'NO_STOCK_OR_ENDED':  'Hết hàng / ngừng bán',
    'STOCKOUT_IMMINENT':  'Sắp hết hàng',
    'STOCKOUT_RISK':      'Nguy cơ hết hàng',
    'STOCKOUT_NOW':       'Đã hết hàng',
}
RISK_VI = {'CRITICAL':'Rất cao','HIGH':'Cao','MEDIUM':'Trung bình','LOW':'Thấp','NO_SALES':'Không phát sinh bán'}

_m  = lambda v: '—' if pd.isna(v) else f'{v:,.0f}'
_pc = lambda v: '—' if pd.isna(v) else f'{v*100:.{PCT_DP}f}%'
_n1 = lambda v: '—' if pd.isna(v) else f'{v:,.1f}'

# ---------------------------------------------------------------- 5.1 tổng quan
alive = sk[(sk.is_inactive != 1)]
tot_stock_val = alive.stock_value_vnd.sum()
tot_excess    = alive.excess_value_vnd.sum()

ov = (alive.groupby('alert_code')
        .agg(so_sku=('variation_code','count'),
             ton_kho=('stock_units','sum'),
             gia_tri_ton=('stock_value_vnd','sum'),
             gia_tri_du=('excess_value_vnd','sum'),
             dt_tuan=('revenue_week_vnd','sum'))
        .reset_index().sort_values('gia_tri_ton', ascending=False))
ov['Cảnh báo']              = ov.alert_code.map(lambda c: ALERT_VI.get(c, c))
ov['Số SKU']                = ov.so_sku.map(lambda v: f'{v:,.0f}')
ov['SL tồn kho']            = ov.ton_kho.map(_m)
ov['Giá trị tồn (VND)']     = ov.gia_tri_ton.map(_m)
ov['Giá trị dư thừa (VND)'] = ov.gia_tri_du.map(_m)
ov['Doanh thu tuần (VND)']  = ov.dt_tuan.map(_m)
ov['% giá trị tồn']         = (ov.gia_tri_ton / tot_stock_val * 100).map(lambda v: f'{v:.{PCT_DP}f}%')

head(f'5.1 Sức khoẻ danh mục — {len(alive):,} SKU đang hoạt động',
     'trạng thái hiện hành (is_latest = 1), phân loại theo alert_code · '
     'không gắn với một khoảng thời gian cụ thể')
table(ov[['Cảnh báo','Số SKU','SL tồn kho','Giá trị tồn (VND)','% giá trị tồn',
          'Giá trị dư thừa (VND)','Doanh thu tuần (VND)']])

banner(f'Tổng giá trị tồn kho theo giá bán: <b>{tot_stock_val/1e9:,.1f} tỷ VND</b>. '
       f'Trong đó phần được đánh giá là dư thừa: <b>{tot_excess/1e9:,.1f} tỷ VND</b> '
       f'(<b>{tot_excess/tot_stock_val*100:.{PCT_DP}f}%</b>) trên '
       f'{int((alive.excess_units.fillna(0) > 0).sum()):,} SKU. '
       f'Đây là tiền đang nằm trong kho, không phải doanh thu đã mất.',
       ORANGE if tot_excess/tot_stock_val > 0.3 else BLUE)

# --------------------------------------------------------- 5.2 bán chạy trong tuần
best = (sk[sk.units_week > 0]
        .sort_values(['revenue_week_vnd','units_week'], ascending=[False,False]).head(10).copy())
if len(best):
    b = pd.DataFrame({
        'Mã SKU':                best.variation_code,
        'Tên sản phẩm':          best.product_name.fillna('(chưa đặt tên)'),
        'Phân loại':             best.variation_attributes.fillna('—'),
        'SL bán (tuần)':         best.units_week.map(_m),
        'Doanh thu tuần (VND)':  best.revenue_week_vnd.map(_m),
        'SL tồn kho':            best.stock_units.map(_m),
        'Số tuần còn hàng':      best.weeks_of_cover.map(_n1),
        'Cảnh báo':              best.alert_code.map(lambda c: ALERT_VI.get(c, c)),
    })
    head(f'5.2 Bán chạy nhất tuần {WK_START} – {WK_END}',
         'xếp theo DOANH THU THẬT của từng SKU (revenue_vn), không phải số lượng nhân đơn giá bình quân')
    table(b)
    _risky = best[best.alert_code.isin(['STOCKOUT_NOW','STOCKOUT_IMMINENT','STOCKOUT_RISK'])]
    if len(_risky):
        banner(f'<b>{len(_risky)} trong 10 SKU bán chạy nhất đang có cảnh báo hết hàng.</b> '
               + '; '.join(f'{r.variation_code} ({ALERT_VI.get(r.alert_code, r.alert_code)}, '
                           f'còn {0 if pd.isna(r.days_until_stockout) else r.days_until_stockout:,.0f} ngày)'
                           for _, r in _risky.iterrows())
               + '. Mất hàng ở nhóm này là mất doanh thu đã được chứng minh.')

# ------------------------------------------------------------- 5.3 nguy cơ hết hàng
# Tiêu chí phụ là avg_daily_sales_30d — một TỐC ĐỘ, so sánh được giữa các SKU.
# Không dùng `revenue` hay `units_sold_day_range`: hai cột đó tính trên `days_range`
# riêng của từng SKU (35 hoặc 60 ngày), nên so sánh trực tiếp giữa các SKU là sai.
soon = (sk[sk.alert_code.isin(['STOCKOUT_NOW','STOCKOUT_IMMINENT','STOCKOUT_RISK'])]
        .sort_values(['restock_urgency','avg_daily_30d'], ascending=[False,False]).head(10).copy())
head('5.3 Nguy cơ hết hàng — cần nhập bổ sung',
     'xếp theo restock_urgency_score, sau đó tốc độ bán 30 ngày; '
     'SL đề xuất nhập lấy từ recommended_restock_qty')
if len(soon):
    s3 = pd.DataFrame({
        'Mã SKU':               soon.variation_code,
        'Tên sản phẩm':         soon.product_name.fillna('(chưa đặt tên)'),
        'SL tồn kho':           soon.stock_units.map(_m),
        'SL bán/ngày (30d)':    soon.avg_daily_30d.map(_n1),
        'Số ngày đến hết hàng': soon.days_until_stockout.map(_n1),
        'Mức rủi ro':           soon.stockout_risk_level.map(lambda v: RISK_VI.get(v, v)),
        'SL đề xuất nhập':      soon.rec_restock_qty.map(_m),
        'Cảnh báo':             soon.alert_code.map(lambda c: ALERT_VI.get(c, c)),
    })
    table(s3)
else:
    bullets([], BLUE, 'SKU có nguy cơ hết hàng')

# --------------------------------------------------- 5.4 tồn kho dư thừa / hàng chết
slow = (sk[sk.alert_code.isin(['OVERSTOCK_CRITICAL','OVERSTOCK_HIGH','OVERSTOCK_WATCH','DEAD_STOCK'])
           & (sk.stock_units.fillna(0) > 0)]
        .sort_values('excess_value_vnd', ascending=False).head(10).copy())
head('5.4 Tồn kho dư thừa và hàng chết — tiền đang nằm trong kho',
     'xếp theo excess_value; % giảm giá lấy từ suggested_discount_pct (bậc 30/40/50)')
if len(slow):
    s4 = pd.DataFrame({
        'Mã SKU':                 slow.variation_code,
        'Tên sản phẩm':           slow.product_name.fillna('(chưa đặt tên)'),
        'SL tồn kho':             slow.stock_units.map(_m),
        'SL dư thừa':             slow.excess_units.map(_m),
        'Giá trị dư thừa (VND)':  slow.excess_value_vnd.map(_m),
        'Sell-through':           slow.sell_through.map(_pc),
        'Số ngày tồn (DOH)':      slow.apply(lambda r: '∞' if r.doh_is_infinite == 1 else _n1(r.doh), axis=1),
        '% giảm giá đề xuất':     slow.suggested_discount_pct.map(lambda v: '—' if pd.isna(v) else f'{v:.0f}%'),
        'Cảnh báo':               slow.alert_code.map(lambda c: ALERT_VI.get(c, c)),
    })
    table(s4)
    banner(f'10 SKU trên giữ <b>{slow.excess_value_vnd.sum()/1e9:,.2f} tỷ VND</b> giá trị dư thừa '
           f'— {slow.excess_value_vnd.sum()/tot_excess*100:.{PCT_DP}f}% tổng dư thừa toàn danh mục.')
else:
    bullets([], ORANGE, 'SKU tồn kho dư thừa')

In [ ]:
# ===================== 5.5–5.9 BẢNG XẾP HẠNG SKU (5 danh sách) =====================
# Thêm MỚI: khối SQL riêng, không sửa STOCK_SQL và không đụng vào CTE nào đang có.
# Cả 5 danh sách dùng CHUNG một nguồn (`dash_base_sku_day`) và CHUNG một mốc thời gian
# đã định nghĩa ở ô tham số — MTD_START/MTD_END và WK_START/WK_END. Không định nghĩa lại.
#
# BỐN ĐỊNH NGHĨA PHẢI NÓI RÕ (nếu không, mỗi người đọc một kiểu)
#
# a) "sale_velocity" — TỰ TÍNH, KHÔNG dùng cột `sale_velocity` của report_stats_product.
#    Cột đó bằng units_sold_day_range / days_range, mà `days_range` là 35 HOẶC 60 ngày
#    tuỳ từng SKU (đã kiểm: khớp 4.393/4.393 dòng). Nghĩa là mỗi SKU đo trên một cửa sổ
#    khác nhau, nên so sánh giữa các SKU là sai. Ở đây:
#         sale_velocity = units_sold_60d / 60.0   (số lượng bán trung bình mỗi ngày)
#    cùng một cửa sổ 60 ngày cho mọi SKU, tính đến MTD_END.
#
# b) "hero SKU" — không có bảng `dim_sku`; chỉ có hàm `fn_dim_sku()` (không tham số) và
#    nó KHÔNG có cờ hero. `report_stats_product` CÓ cờ `is_all_time_top_10` (đúng 10 SKU).
#    Nhưng cờ đó lệch khá nhiều với thực tế doanh thu: đo trên dữ liệu đến 30/08 chỉ
#    6/10 trùng nhau. Bốn SKU hạng 7–10 theo doanh thu KHÔNG mang cờ (trong đó
#    6159707906735 đứng hạng 9), còn HANNIE TOP và SIMI TOP mang cờ nhưng chỉ hạng 11–12.
#    Vì yêu cầu là "current year", hero ở đây định nghĩa RÕ RÀNG là:
#         top 10 theo TỔNG DOANH THU THUẦN từ 01/01 đến MTD_END
#    và cột `is_all_time_top_10` vẫn hiển thị bên cạnh để đối chiếu hai cách nhìn.
#
# c) "slow-moving" — số lượng bán THẤP NHẤT trong 90 ngày gần nhất, CHỈ xét SKU còn tồn
#    kho (sl_ton_kho > 0). Không lọc tồn kho thì bảng sẽ đầy SKU đã ngừng bán, hết hàng —
#    đúng về mặt số học nhưng không hành động được. Đồng hạng thì SKU ôm nhiều tiền hơn
#    (giá trị tồn cao hơn) xếp trước.
#
# d) Danh sách 1 là THÔNG TIN THAM KHẢO cho HĐQT khi bàn mức chiết khấu. Nó KHÔNG tự
#    tính % chiết khấu đề xuất. Cột `revenue_share_pct` = doanh thu MTD của SKU chia
#    tổng doanh thu MTD toàn danh mục, để thấy mức độ tập trung. Mức chiết khấu theo SKU
#    đã có sẵn ở mục 5.4 (`suggested_discount_pct`).

STOCK_RANK_SQL = """
WITH
sku_day AS (   -- nguồn duy nhất cho cả 5 danh sách
  SELECT variation_code, period_date,
         IFNULL(units_sold, 0) AS units_sold,
         IFNULL(revenue_vn, 0) AS revenue_vn
  FROM `{FQ}.dash_base_sku_day`
),
dim AS (       -- fn_dim_sku() không có tham số; gộp phòng trường hợp trùng mã
  SELECT variation_code,
         ANY_VALUE(product_name) AS dim_name,
         ANY_VALUE(category)     AS category,
         ANY_VALUE(sub_category) AS sub_category
  FROM `{FQ}.fn_dim_sku`()
  WHERE product_name IS NOT NULL
  GROUP BY variation_code
),
stats AS (
  SELECT variation_code, product_name AS stats_name,
         IFNULL(CAST(sl_ton_kho AS FLOAT64),0) AS stock_units,
         IFNULL(CAST(tong_gia_tri_hang_ton_kho_theo_gia_ban AS FLOAT64),0) AS stock_value_vnd,
         is_all_time_top_10
  FROM `{FQ}.report_stats_product`
  WHERE is_latest = 1
  AND product_name IS NOT NULL
),
nm AS (        -- một tên hiển thị, ưu tiên tên trong dim
  SELECT variation_code,
         COALESCE(d.dim_name, s.stats_name, 'n/a') AS sku_name,
         d.category, d.sub_category, s.stock_units, s.stock_value_vnd,
         IFNULL(s.is_all_time_top_10, FALSE) AS is_all_time_top_10
  FROM dim d FULL OUTER JOIN stats s USING (variation_code)
),

/* ---------- (1) Top 10 doanh thu — THÁNG HIỆN TẠI (month-to-date) --------------
   Cửa sổ: MTD_START..MTD_END, tức đúng logic "days-elapsed" đã dùng ở nơi khác.
   Xếp hạng: tổng revenue_vn giảm dần. Danh sách THAM KHẢO cho quyết định chiết
   khấu của HĐQT — bản thân nó không đề xuất mức %. revenue_share_pct cho thấy
   mức độ tập trung doanh thu.                                                  */
r1_mtd_revenue AS (
  SELECT variation_code AS sku, SUM(revenue_vn) AS mtd_revenue
  FROM sku_day WHERE period_date BETWEEN '{mtd_s}' AND '{mtd_e}'
  GROUP BY 1 HAVING SUM(revenue_vn) > 0
),
r1 AS (
  SELECT sku, mtd_revenue,
         SAFE_DIVIDE(mtd_revenue, SUM(mtd_revenue) OVER ()) AS revenue_share_pct,
         ROW_NUMBER() OVER (ORDER BY mtd_revenue DESC) AS rk
  FROM r1_mtd_revenue
),

/* ---------- (2) Top 10 theo sale velocity — 60 NGÀY GẦN NHẤT ------------------
   Cửa sổ: 60 ngày tính ĐẾN MTD_END (bao gồm cả hai đầu).
   sale_velocity = units_sold_60d / 60.0  -> số lượng bán trung bình MỖI NGÀY.
   Xếp hạng theo velocity giảm dần; vì mẫu số là hằng số 60 nên thứ tự trùng với
   thứ tự theo units_sold_60d — velocity giữ lại vì đó là đơn vị đọc được.       */
r2_velocity AS (
  SELECT variation_code AS sku,
         SUM(units_sold)        AS units_sold_60d,
         SUM(units_sold) / 60.0 AS sale_velocity
  FROM sku_day
  WHERE period_date BETWEEN DATE_SUB(DATE '{mtd_e}', INTERVAL 59 DAY) AND '{mtd_e}'
  GROUP BY 1 HAVING SUM(units_sold) > 0
),
r2 AS (
  SELECT *, ROW_NUMBER() OVER (ORDER BY sale_velocity DESC, units_sold_60d DESC) AS rk
  FROM r2_velocity
),

/* ---------- (3) Top 10 "hero" — NĂM HIỆN TẠI ----------------------------------
   hero = 10 SKU có TỔNG DOANH THU THUẦN cao nhất từ 01/01 đến MTD_END.
   Cột is_all_time_top_10 của report_stats_product hiển thị kèm để đối chiếu;
   hai cách nhìn lệch nhau 2/10 nên không dùng cờ đó làm tiêu chí xếp hạng.      */
r3_ytd AS (
  SELECT variation_code AS sku, SUM(revenue_vn) AS ytd_revenue, SUM(units_sold) AS ytd_units
  FROM sku_day
  WHERE period_date BETWEEN DATE_TRUNC(DATE '{mtd_e}', YEAR) AND '{mtd_e}'
  GROUP BY 1 HAVING SUM(revenue_vn) > 0
),
r3 AS (
  SELECT *, ROW_NUMBER() OVER (ORDER BY ytd_revenue DESC) AS rk FROM r3_ytd
),
hero_10 AS (SELECT sku FROM r3 WHERE rk <= 10),   -- dùng để loại trừ ở (4a)/(4b)

/* ---------- (4a) Top 10 doanh thu TUẦN, loại trừ hero ------------------------
   Cửa sổ: WK_START..WK_END (giữ nguyên quy ước tuần bắt đầu Thứ Hai).
   Mục đích: nhìn ai đang NỔI LÊN, tách khỏi nhóm hero đã ổn định của cả năm.    */
r4a_wtd AS (
  SELECT variation_code AS sku, SUM(revenue_vn) AS wtd_revenue
  FROM sku_day WHERE period_date BETWEEN '{ws_d}' AND '{we_d}'
  GROUP BY 1 HAVING SUM(revenue_vn) > 0
),
r4a AS (
  SELECT sku, wtd_revenue, ROW_NUMBER() OVER (ORDER BY wtd_revenue DESC) AS rk
  FROM r4a_wtd WHERE sku NOT IN (SELECT sku FROM hero_10)
),

/* ---------- (4b) Top 10 doanh thu THÁNG (MTD), loại trừ hero ----------------- */
r4b AS (
  SELECT sku, mtd_revenue, ROW_NUMBER() OVER (ORDER BY mtd_revenue DESC) AS rk
  FROM r1_mtd_revenue WHERE sku NOT IN (SELECT sku FROM hero_10)
),

/* ---------- (5) Top 10 SKU bán chậm nhất THEO TỪNG category/sub_category -----
   slow-moving = số lượng bán thấp nhất trong 90 ngày gần nhất, chỉ xét SKU CÒN
   TỒN KHO (stock_units > 0). Đồng hạng: SKU ôm nhiều tiền hơn xếp trước.
   ROW_NUMBER() PARTITION BY category, sub_category -> top 10 TRONG TỪNG NHÓM,
   không phải top 10 toàn bộ.                                                    */
u90 AS (
  SELECT variation_code AS sku, SUM(units_sold) AS units_sold_90d
  FROM sku_day
  WHERE period_date BETWEEN DATE_SUB(DATE '{mtd_e}', INTERVAL 89 DAY) AND '{mtd_e}'
  GROUP BY 1
),
r5 AS (
  SELECT nm.category, nm.sub_category, nm.variation_code AS sku,
         IFNULL(u90.units_sold_90d, 0) AS units_sold_90d,
         nm.stock_units, nm.stock_value_vnd,
         ROW_NUMBER() OVER (PARTITION BY nm.category, nm.sub_category
                            ORDER BY IFNULL(u90.units_sold_90d, 0) ASC,
                                     nm.stock_value_vnd DESC) AS rank_within_group
  FROM nm LEFT JOIN u90 ON u90.sku = nm.variation_code
  WHERE nm.stock_units > 0 AND nm.category IS NOT NULL
)
,master AS (
SELECT '1_mtd_revenue' AS list_id, r1.rk AS rank, CAST(NULL AS INT64) AS rank_within_group,
       nm.category, nm.sub_category, r1.sku, nm.sku_name,
       r1.mtd_revenue, CAST(NULL AS FLOAT64) AS wtd_revenue, CAST(NULL AS FLOAT64) AS ytd_revenue,
       CAST(NULL AS FLOAT64) AS units_sold_60d, CAST(NULL AS FLOAT64) AS sale_velocity,
       CAST(NULL AS FLOAT64) AS units_sold_90d, r1.revenue_share_pct,
       nm.is_all_time_top_10, CAST(NULL AS FLOAT64) AS stock_units
FROM r1 LEFT JOIN nm ON nm.variation_code = r1.sku WHERE r1.rk <= 10
UNION ALL
SELECT '2_velocity_60d', r2.rk, NULL, nm.category, nm.sub_category, r2.sku, nm.sku_name,
       NULL, NULL, NULL, r2.units_sold_60d, r2.sale_velocity, NULL, NULL,
       nm.is_all_time_top_10, NULL
FROM r2 LEFT JOIN nm ON nm.variation_code = r2.sku WHERE r2.rk <= 10
UNION ALL
SELECT '3_hero_ytd', r3.rk, NULL, nm.category, nm.sub_category, r3.sku, nm.sku_name,
       NULL, NULL, r3.ytd_revenue, NULL, NULL, NULL, NULL,
       nm.is_all_time_top_10, NULL
FROM r3 LEFT JOIN nm ON nm.variation_code = r3.sku WHERE r3.rk <= 10
UNION ALL
SELECT '4a_wtd_ex_hero', r4a.rk, NULL, nm.category, nm.sub_category, r4a.sku, nm.sku_name,
       NULL, r4a.wtd_revenue, NULL, NULL, NULL, NULL, NULL,
       nm.is_all_time_top_10, NULL
FROM r4a LEFT JOIN nm ON nm.variation_code = r4a.sku WHERE r4a.rk <= 10
UNION ALL
SELECT '4b_mtd_ex_hero', r4b.rk, NULL, nm.category, nm.sub_category, r4b.sku, nm.sku_name,
       r4b.mtd_revenue, NULL, NULL, NULL, NULL, NULL, NULL,
       nm.is_all_time_top_10, NULL
FROM r4b LEFT JOIN nm ON nm.variation_code = r4b.sku WHERE r4b.rk <= 10
UNION ALL
SELECT '5_slow_by_category', NULL, r5.rank_within_group, r5.category, r5.sub_category,
       r5.sku, nm.sku_name, NULL, NULL, NULL, NULL, NULL, r5.units_sold_90d, NULL,
       nm.is_all_time_top_10, r5.stock_units
FROM r5 LEFT JOIN nm ON nm.variation_code = r5.sku WHERE r5.rank_within_group <= 10
)
SELECT * FROM master WHERE sku_name is not null AND category NOT LIKE 'NON-RETAIL'
"""

rk = q(STOCK_RANK_SQL, mtd_s=MTD_START, mtd_e=MTD_END, ws_d=WK_START, we_d=WK_END)
_L = lambda t: rk[rk.list_id == t].copy()
def _hero_mark(v):
    return '★' if pd.notna(v) and bool(v) else ''

# ------------------------------------------------------------------ 5.5 (1)
d = _L('1_mtd_revenue').sort_values('rank')
head(f'5.5 Top 10 SKU theo doanh thu — tháng hiện tại ({MTD_START} – {MTD_END}, {N_DAYS} ngày)',
     'THAM KHẢO cho quyết định chiết khấu của HĐQT — danh sách này không tự đề xuất mức %. '
     'Mức % theo từng SKU nằm ở mục 5.4.')
table(pd.DataFrame({
    'Hạng': d['rank'], 'Mã SKU': d.sku, 'Tên sản phẩm': d.sku_name,
    'Ngành hàng': d.category.fillna('—'), 'Nhóm hàng': d.sub_category.fillna('—'),
    'Doanh thu MTD (VND)': d.mtd_revenue.map(_m),
    '% trên tổng doanh thu MTD': d.revenue_share_pct.map(_pc),
    'Hero': d.is_all_time_top_10.map(_hero_mark)}))

# ------------------------------------------------------------------ 5.6 (2)
d = _L('2_velocity_60d').sort_values('rank')
head('5.6 Top 10 SKU theo tốc độ bán — 60 ngày gần nhất',
     f'sale_velocity = units_sold_60d / 60 (SL bán trung bình mỗi ngày), cửa sổ tính đến {MTD_END}. '
     'Không dùng cột sale_velocity của report_stats_product vì cột đó đo trên 35 hoặc 60 ngày '
     'tuỳ từng SKU nên không so sánh được với nhau.')
table(pd.DataFrame({
    'Hạng': d['rank'], 'Mã SKU': d.sku, 'Tên sản phẩm': d.sku_name,
    'Nhóm hàng': d.sub_category.fillna('—'),
    'SL bán 60 ngày': d.units_sold_60d.map(_m),
    'Tốc độ bán (SL/ngày)': d.sale_velocity.map(_n1),
    'Hero': d.is_all_time_top_10.map(_hero_mark)}))

# ------------------------------------------------------------------ 5.7 (3)
d = _L('3_hero_ytd').sort_values('rank')
_agree = int(d.is_all_time_top_10.sum())
head(f'5.7 Top 10 SKU "hero" — năm hiện tại (01/01 – {MTD_END})',
     'hero = doanh thu thuần luỹ kế năm cao nhất. Cột cuối là cờ is_all_time_top_10 có sẵn '
     'trong report_stats_product, để đối chiếu chứ không dùng làm tiêu chí xếp hạng.')
table(pd.DataFrame({
    'Hạng': d['rank'], 'Mã SKU': d.sku, 'Tên sản phẩm': d.sku_name,
    'Ngành hàng': d.category.fillna('—'), 'Nhóm hàng': d.sub_category.fillna('—'),
    'Doanh thu YTD (VND)': d.ytd_revenue.map(_m),
    'Có cờ is_all_time_top_10': d.is_all_time_top_10.map(
    lambda v: 'Có' if pd.notna(v) and bool(v) else 'Không'
)}))
banner(f'{_agree}/10 SKU trong danh sách này cũng mang cờ <b>is_all_time_top_10</b> sẵn có. '
       f'{10 - _agree} SKU còn lại lệch nhau giữa hai cách xác định — bảng trên xếp theo doanh thu '
       f'thực tế trong năm, cờ kia tính theo logic riêng của report_stats_product.',
       BLUE if _agree >= 8 else ORANGE)

# ------------------------------------------------------------------ 5.8 (4a/4b)
for lid, col, lbl, note in [
    ('4a_wtd_ex_hero', 'wtd_revenue', f'tuần {WK_START} – {WK_END}', 'Doanh thu tuần (VND)'),
    ('4b_mtd_ex_hero', 'mtd_revenue', f'tháng {MTD_START} – {MTD_END}', 'Doanh thu MTD (VND)')]:
    d = _L(lid).sort_values('rank')
    head(f'5.8 Top 10 SKU {lbl} — ĐÃ LOẠI 10 hero của năm',
         'so sánh nhóm đang nổi lên với nhóm hero đã ổn định; hero lấy từ mục 5.7')
    table(pd.DataFrame({
        'Hạng': d['rank'], 'Mã SKU': d.sku, 'Tên sản phẩm': d.sku_name,
        'Ngành hàng': d.category.fillna('—'), 'Nhóm hàng': d.sub_category.fillna('—'),
        note: d[col].map(_m)}))

# ------------------------------------------------------------------ 5.9 (5)
d = _L('5_slow_by_category')
head('5.9 Top 10 SKU bán chậm nhất trong TỪNG ngành hàng / nhóm hàng',
     'bán chậm = SL bán thấp nhất trong 90 ngày gần nhất, chỉ xét SKU còn tồn kho; '
     'đồng hạng thì SKU ôm nhiều tiền hơn xếp trước. Xếp hạng riêng trong mỗi nhóm.')
if len(d):
    d = d.sort_values(['category','sub_category','rank_within_group'])
    table(pd.DataFrame({
        'Ngành hàng': d.category, 'Nhóm hàng': d.sub_category,
        'Hạng trong nhóm': d.rank_within_group.astype('Int64'),
        'Mã SKU': d.sku, 'Tên sản phẩm': d.sku_name,
        'SL bán 90 ngày': d.units_sold_90d.map(_m),
        'SL tồn kho': d.stock_units.map(_m)}))
    banner(f'{d.sub_category.nunique()} nhóm hàng có SKU bán chậm còn tồn kho, '
           f'tổng {len(d):,} dòng được liệt kê (tối đa 10 mỗi nhóm).')
else:
    bullets([], ORANGE, 'SKU bán chậm')

---
# 6. Action items & kế hoạch doanh thu

In [ ]:
# Hành động hiệu suất. Từ 2026-09-06 nguồn chính là `dash_action_items.action_item_vi` —
# tiếng Việt do gemini-3.5-flash viết, đã qua 5 chốt kiểm tra tất định trong chính bảng đó
# (số liệu khớp câu SQL, mã SKU còn nguyên, không có câu nhân quả, có dấu tiếng Việt, đủ dài).
#
# ĐÃ ĐỔI: trước đây ô này lấy phần đuôi ACTIONS của `narrative_vi`, tức là nhờ model của
# trang dịch lại. Khi tầng AI bị chặn (quarantine) thì phần đuôi đó rỗng và báo cáo rơi về
# `action_item` TIẾNG ANH. Nay bản tiếng Việt luôn tồn tại trong bảng và không phụ thuộc vào
# việc trang có được sinh hay không.
#
# Bảng kế hoạch doanh thu ở ô sau được dựng ở **level_2**, cấp mà chỉ tiêu thực sự tồn tại.
# Công thức:
# - **cần đạt tuần tới** = chỉ tiêu tuần kế tiếp
# - **cần đạt mỗi tuần còn lại trong tháng** = `(chỉ tiêu tháng − NMV luỹ kế tháng) / số tuần còn lại`
# - **cần đạt mỗi tuần còn lại trong năm** = `(chỉ tiêu năm − NMV luỹ kế năm) / số tuần còn lại`
# - **AT RISK** khi mức cần đạt vượt 130% doanh thu trung bình 4 tuần gần nhất
for page, label in [('PAGE1_WOW', 'Tuần này so với tuần trước'),
                    ('PAGE2_MTD', 'Luỹ kế tháng so với chỉ tiêu')]:
    d = perf[perf.page == page]
    if not len(d):
        continue

    items = [f'{action}'
             f'<div style="font:400 11.5px system-ui;color:{GREY};margin-top:5px">'
             f'Dữ liệu từ {r.source_item}</div>'
             for _, r in d.iterrows()
             for action in split_actions(r.action_item_vi or r.action_item)]

    _n = int((d.vi_source == 'ai_vi').sum())
    src_note = (f'Dữ liệu từ  `dash_action_items` — số liệu được tính toán sẵn, văn bản tạo bởi gemini-3.5-flash '
                f'({_n}/{len(d)} dòng qua đủ 5 mục kiểm tra độ chính xác của AI)')

    head(f'{label} — hành động', src_note)
    bullets(items, ORANGE, 'hành động')

    _fallback = d[d.vi_source != 'ai_vi']
    if len(_fallback):
        banner(f'<b>{len(_fallback)} hành động</b> đang hiển thị tiếng Anh vì bản do '
               f'nội dung AI model viết không đạt yêu cầu sau khi kiểm tra ('
               + ', '.join(sorted(set(_fallback.gen_status))) + ').', ORANGE)


In [ ]:
RUNRATE_SQL = """
WITH act AS (
  SELECT level_2,
    SUM(IF(period_date BETWEEN '{ws_d}' AND '{we_d}', g, 0))                         AS nmv_week,
    SUM(IF(period_date BETWEEN DATE_TRUNC(DATE '{we_d}', MONTH) AND '{we_d}', g, 0)) AS nmv_mtd,
    SUM(IF(period_date BETWEEN DATE_TRUNC(DATE '{we_d}', YEAR)  AND '{we_d}', g, 0)) AS nmv_ytd,
    SUM(IF(period_date BETWEEN DATE_SUB(DATE '{ws_d}', INTERVAL 21 DAY) AND '{we_d}', g, 0))/4 AS avg_4wk
  FROM (SELECT level_2, period_date,
               IF(IS_NAN(nmv) OR nmv IS NULL, 0, nmv) AS g   -- NaN guard
        FROM `{FQ}.dash_base_order_day`)
  WHERE level_2 NOT LIKE '%Wholesales%'
  GROUP BY level_2
),
tgt AS (
  -- week/month/year_target_revenue are already the period total per level_2, repeated on
  -- every matching date row — SUM() over those rows would multiply the target by the row
  -- count (~7x/~30x/~365x), so take MAX() to read the constant value instead.
  SELECT level_2,
    MAX(IF(sale_date BETWEEN DATE_ADD(DATE '{we_d}', INTERVAL 1 DAY)
                         AND DATE_ADD(DATE '{we_d}', INTERVAL 7 DAY), ROUND(week_target_revenue,0), 0)) AS tgt_next_week,
    MAX(IF(DATE_TRUNC(sale_date, MONTH) = DATE_TRUNC(DATE '{we_d}', MONTH), month_target_revenue, 0)) AS tgt_month,
    MAX(IF(DATE_TRUNC(sale_date, YEAR)  = DATE_TRUNC(DATE '{we_d}', YEAR),  year_target_revenue, 0)) AS tgt_year
  FROM `{FQ}.dash_target_day`
  WHERE level_2 NOT LIKE '%Wholesales%'
  GROUP BY 1
)
SELECT a.*, t.tgt_next_week, t.tgt_month, t.tgt_year,
  DATE_DIFF(LAST_DAY(DATE '{we_d}'), DATE '{we_d}', DAY)/7.0 AS wks_left_month,
  DATE_DIFF(DATE(EXTRACT(YEAR FROM DATE '{we_d}'),12,31), DATE '{we_d}', DAY)/7.0 AS wks_left_year
FROM act a LEFT JOIN tgt t ON t.level_2 = a.level_2
"""
rr = q(RUNRATE_SQL, ws_d=WK_START, we_d=WK_END)

def need(target, actual, weeks):
    if pd.isna(target) or target == 0 or weeks is None or weeks <= 0: return None
    return max(target - actual, 0) / weeks

out = []
for _, r in rr.iterrows():
    n_wk  = r.tgt_next_week if (pd.notna(r.tgt_next_week) and r.tgt_next_week != 0) else None
    n_mth = need(r.tgt_month, r.nmv_mtd, r.wks_left_month)
    n_yr  = need(r.tgt_year,  r.nmv_ytd, r.wks_left_year)
    worst = max([x for x in (n_wk, n_mth, n_yr) if x is not None], default=None)
    has_target = any(pd.notna(x) and x != 0 for x in (r.tgt_next_week, r.tgt_month, r.tgt_year))
    if not has_target:      status = 'no target'
    elif not worst or not r.avg_4wk: status = '—'
    else: status = 'AT RISK' if worst > RISK_THRESHOLD * r.avg_4wk else 'ok'
    out.append({'channel': f'{r.level_2}',
      'nmv_actual_week_vnd':   f'{r.nmv_week:,.0f}',
      'need_next_week_vnd':    f'{n_wk:,.0f}'  if n_wk  is not None else 'no target',
      'need_per_wk_month_vnd': f'{n_mth:,.0f}' if n_mth is not None else 'no target',
      'need_per_wk_year_vnd':  f'{n_yr:,.0f}'  if n_yr  is not None else 'no target',
      'trailing_4wk_avg_vnd':  f'{r.avg_4wk:,.0f}',
      'required_vs_4wk_pct':   f'{worst/r.avg_4wk*100:.{PCT_DP}f}%' if (worst and r.avg_4wk) else '—',
      'status': status, '_s': r.nmv_week})

rrt = pd.DataFrame(out).sort_values('_s', ascending=False).drop(columns='_s')

# Rename specified columns
rrt = rrt.rename(columns={
    'channel': 'Kênh bán',
    'nmv_actual_week_vnd': 'Doanh thu thuần (Tuần)',
    'need_next_week_vnd': 'Doanh thu cần đạt trong tuần tới',
    'need_per_wk_month_vnd': 'Doanh thu cần đạt mỗi tuần trong tháng',
    'need_per_wk_year_vnd': 'Doanh thu cần đạt mỗi tuần trong năm để đủ target 12M',
    'trailing_4wk_avg_vnd': 'Doanh thu trung bình 4 tuần trước',
    'required_vs_4wk_pct': 'Tỷ lệ cần đạt so với 4 tuần trước'
})

hl = lambda v: f'background-color:#FFF1E2;color:{ORANGE};font-weight:600' if v=='AT RISK' else ''
head(f'Kế hoạch doanh thu theo từng kênh',
     f'at risk = tỷ lệ cần đạt > {RISK_THRESHOLD:.0%} of so với trung bình doanh thu 4 tuần trước')
display(rrt.style.hide(axis='index').map(hl, subset=['status']).set_table_styles(TH))

at_risk = rrt[rrt.status=='AT RISK']
if len(at_risk):
    banner(f'<b>{len(at_risk)} kênh bán trong tình trạng AT RISK (Cần chú ý).</b> % cần đạt khoảng cách '
           f'vượt hơn {RISK_THRESHOLD:.0%} của doanh thu đã đạt từ tuần trước '
           f'4 tuần. Cần xem lại kế hoạch (target) các kênh này vì target đang vượt xa doanh thu thực tại.')

In [ ]:
# Chi tiết theo level_3. `dash_trend_weekly` nay mang `week_target` — chỉ tiêu level_2 được
# phân bổ xuống level_3 theo tỷ trọng NMV 8 tuần gần nhất (tổng tỷ trọng = 1, nên cộng lại
# đúng bằng chỉ tiêu level_2). Ghi chú cũ trong notebook nói không thể có chỉ tiêu ở level_3
# — điều đó không còn đúng.
l3 = q("""
SELECT level_1, level_2, IFNULL(level_3,'(unattributed)') AS level_3,
       SUM(nmv)                       AS nmv_week_vnd,
       SUM(bills)                     AS bills,
       SUM(week_target)               AS week_target_vnd,
       SAFE_DIVIDE(SUM(nmv), NULLIF(SUM(week_target),0)) AS attainment,
       ANY_VALUE(target_alloc_method) AS alloc_method
FROM `{FQ}.dash_trend_weekly`
WHERE week_start = DATE '{ws_d}'
GROUP BY 1,2,3
ORDER BY nmv_week_vnd DESC
""", ws_d=WK_START)

l3['nmv_vnd']    = l3.nmv_week_vnd.map(lambda v: f'{v:,.0f}')
l3['target_vnd'] = l3.week_target_vnd.map(lambda v: '—' if pd.isna(v) or v == 0 else f'{v:,.0f}')
l3['attain']     = l3.attainment.map(lambda v: '—' if pd.isna(v) else f'{v*100:.{PCT_DP}f}%')
l3['share']      = (l3.nmv_week_vnd / l3.nmv_week_vnd.sum()*100).map(lambda v: f'{v:.{PCT_DP}f}%')

head('Kênh bán chi tiết — doanh thu và chỉ tiêu trong tuần',
     'chỉ tiêu level_3 là phần phân bổ từ chỉ tiêu level_2 theo tỷ trọng NMV 8 tuần gần nhất')

tbl_df = l3[['level_3','nmv_vnd','target_vnd','attain','bills','share']].rename(columns={
    'level_3': 'Kênh bán',
    'nmv_vnd': 'Doanh thu thuần (tuần)',
    'target_vnd': 'Chỉ tiêu tuần (phân bổ)',
    'attain': 'Đạt chỉ tiêu',
    'bills': 'SL đơn',
    'share': 'Tỷ trọng'
})

def _hl_attain(v):
    if not isinstance(v,str) or v == '—': return ''
    try:
        n = float(v.replace('%',''))
        return (f'color:{GREEN};font-weight:600' if n >= 100
                else f'color:{RED};font-weight:600' if n < 80 else '')
    except ValueError:
        return ''

display(tbl_df.style.hide(axis='index')
        .map(_hl_attain, subset=['Đạt chỉ tiêu'])
        .set_table_styles(TH))